This notebook was ran on L4 GPU with high RAM. Due to inconsistencies in the PPMI web interface, we implemented custom monkey-patches to the scraping logic (https://github.com/LivingPark-MRI/ppmi-scraper). In cases where automated retrieval failed, data were manually downloaded and verified to ensure dataset completeness.

# File Loading and Credentials

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Define paths
metadata_path = '/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/T1_metadata_multilcmm_longitudinal_3corrected.csv'
checkpoint_path = '/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/processing_checkpoint_multilcmm_longitudinal.csv'

# Load files into memory
df_metadata = pd.read_csv(metadata_path)
df_checkpoint = pd.read_csv(checkpoint_path)

# Define PPMI credentials
MY_EMAIL = ""
MY_PASSWORD = ""

In [ ]:
num_pending_patients = df_checkpoint[df_checkpoint['status'] == 'pending'].shape[0]
print(f"Number of patients with 'pending' status: {num_pending_patients}")

Number of patients with 'pending' status: 0


# PPMI Downloader Setup and Patching

This entire section runs in ~1 min

In [ ]:
pip install ppmi-downloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.7/109.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 18.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
!add-apt-repository -y ppa:saiarcot895/chromium-beta
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver

PPA publishes dbgsym, you may need to include 'main/debug' component
Repository: 'deb https://ppa.launchpadcontent.net/saiarcot895/chromium-beta/ubuntu/ jammy main'
Description:
This PPA contains the latest Chromium Beta builds, with hardware video decoding enabled (hidden behind a flag), and support for Widevine (needed for viewing many DRM-protected videos) enabled.

== Hardware Video Decoding ==

To enable hardware video decoding, start Chromium with the --enable-features=VaapiVideoDecoder argument. To make this persistent, create a file at /etc/chromium-browser/customizations/92-vaapi-hardware-decoding with the following contents:

CHROMIUM_FLAGS="${CHROMIUM_FLAGS} --enable-features=VaapiVideoDecoder"

See also https://wiki.archlinux.org/title/Chromium#Hardware_video_acceleration for more information on VAAPI video decoding support.

=== Widevine Support ===

The packages in this PPA have support for Widevine inside Chromium enabled. However, you still need to copy some files from 

In [ ]:
!dpkg -L chromium-chromedriver
!ls -l /usr/lib/chromium-browser/chromedriver
!ln -sf /usr/lib/chromium-browser/chromedriver /usr/bin/chromedriver
!chromedriver --version

/.
/usr
/usr/lib
/usr/lib/chromium-browser
/usr/lib/chromium-browser/chromedriver
/usr/share
/usr/share/doc
/usr/share/doc/chromium-chromedriver
/usr/share/doc/chromium-chromedriver/changelog.Debian.gz
/usr/share/doc/chromium-chromedriver/copyright
-rwxr-xr-x 1 root root 16036344 Nov 14  2022 /usr/lib/chromium-browser/chromedriver
ChromeDriver 108.0.5359.40 (280b5fcaab3e877562b06cfaf2eb51121e13c3b9-refs/branch-heads/5359@{#689})


In [ ]:
import shutil
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
import ppmi_downloader.ppmi_downloader

# Re-define the patch to ensure it uses the newly installed binaries
def get_driver_final(headless: bool, tempdir: str, remote: str = None):
    options = webdriver.ChromeOptions()
    prefs = {
        "download.default_directory": tempdir,
        "download.prompt_for_download": False,
    }
    options.add_experimental_option("prefs", prefs)

    # Necessary flags
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    # Use the system installed chromium-browser
    chrome_bin = shutil.which("chromium-browser")
    if chrome_bin:
        options.binary_location = chrome_bin
        print(f"Using Chrome binary at: {chrome_bin}")

    if remote is None:
        # Use the explicitly linked chromedriver
        service = Service(executable_path="/usr/bin/chromedriver")
        driver = webdriver.Chrome(service=service, options=options)
    else:
        if remote == "hostname":
            remote = ppmi_downloader.ppmi_downloader.get_ip_hostname()
        options.add_argument("--ignore-ssl-errors=yes")
        options.add_argument("--ignore-certificate-errors")
        driver = None
        with ppmi_downloader.ppmi_downloader.timeout_manager(30):
            driver = webdriver.Remote(remote, options=options)

    driver.set_window_size(1920, 1080)
    driver.maximize_window()
    return driver

# Apply the patch
ppmi_downloader.ppmi_downloader.get_driver = get_driver_final

In [ ]:
import ppmi_downloader
import os

CONFIG_FILE = ".ppmi_config_auto"

# Create the configuration file in INI format expected by ConfigParser
# The library expects the key 'login' rather than 'email'
with open(CONFIG_FILE, 'w') as f:
    f.write("[ppmi]\n")
    f.write(f"login = {MY_EMAIL}\n")
    f.write(f"password = {MY_PASSWORD}\n")

print(f"Configuration file '{CONFIG_FILE}' created with correct section header and keys.")

# Initialize PPMIDownloader using the config file
try:
    # We still rely on the 'get_driver' patch from the previous cell to handle Chrome
    ppmi = ppmi_downloader.PPMIDownloader(config_file=CONFIG_FILE, headless=True)
    print("SUCCESS: PPMIDownloader initialized using credentials from config file.")
except Exception as e:
    print(f"FAILURE: {e}")

Configuration file '.ppmi_config_auto' created with correct section header and keys.
Using Chrome binary at: /usr/bin/chromium-browser
SUCCESS: PPMIDownloader initialized using credentials from config file.


LivingPark-utils|DEBUG|ppmi_downloader.py:173 in __init__()
                       self.driver.capabilities: {'acceptInsecureCerts': False,
                                                  'browserName': 'chrome',
                                                  'browserVersion': '108.0.5359.40',
                                                  'chrome': {'chromedriverVersion': '108.0.5359.40 '
                                                                                    '(280b5fcaab3e877562b06cfaf2eb51121e13c3b9-refs/branch-heads/5359@{#689})',
                                                             'userDataDir': '/tmp/.org.chromium.Chromium.6dQlCf'},
                                                  'goog:chromeOptions': {'debuggerAddress': 'localhost:42917'},
                                                  'networkConnectionEnabled': False,
                                                  'pageLoadStrategy': 'normal',
                                               

In [ ]:
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

def scroll_and_select(self, cohort):
    """Scrolls through the export table and selects rows matching the cohort metadata."""
    print("Selecting target scans via scrolling...")
    target_metadata = set((str(row['PATNO']), str(row['EVENT_ID']), str(row['Description'])) for _, row in cohort.iterrows())
    found_metadata = set()
    prev_row_count = 0
    stale_scrolls = 0

    while found_metadata != target_metadata:
        try:
            WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.ID, 'tableData')))
            rows = self.driver.find_elements(By.XPATH, "//div[@id='tableData']//tr")

            for row in rows:
                try:
                    cells = row.find_elements(By.TAG_NAME, 'td')
                    if len(cells) > 6:
                        meta = (cells[0].text.strip(), cells[4].text.strip(), cells[6].text.strip())
                        if meta in target_metadata and meta not in found_metadata:
                            checkbox = row.find_element(By.NAME, 'checkbox')
                            if not checkbox.is_selected():
                                self.driver.execute_script('arguments[0].click();', checkbox)
                            found_metadata.add(meta)
                            print(f'  Checked: {meta}')
                except StaleElementReferenceException:
                    continue

            if found_metadata == target_metadata:
                break

            # Attempt to scroll down using the table's arrow button
            scroll_buttons = self.driver.find_elements(By.XPATH, "//tbody[.//div[@id='slider']]//input[@alt='V' and @onclick]")
            if len(scroll_buttons) > 1:
                self.driver.execute_script('arguments[0].click();', scroll_buttons[1])
                time.sleep(1.0) # Wait for table to refresh data
            else:
                stale_scrolls += 5

            if len(rows) == prev_row_count:
                stale_scrolls += 1
            else:
                stale_scrolls = 0

            if stale_scrolls > 15:
                break

            prev_row_count = len(rows)
        except StaleElementReferenceException:
            continue

    if found_metadata != target_metadata:
        # Capture screenshot on failure before raising the error
        self.driver.save_screenshot("debug_scroller_failed.png")
        missing = target_metadata - found_metadata
        raise RuntimeError(f'Could only find {len(found_metadata)} of {len(target_metadata)} scans. Missing: {missing}')

    print('All target scans successfully selected.')

# Monkey-patch the method into the class
ppmi_downloader.PPMIDownloader.scroll_and_select = scroll_and_select
print('Method scroll_and_select (with failure screenshots) has been added.')

Method scroll_and_select (with failure screenshots) has been added.


In [ ]:
def download_all_zip_files(self):
    """Finds and clicks all valid zip links and the metadata link."""
    # Wait for the overlay to show at least one download link
    try:
        WebDriverWait(self.driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[data-download-link='true']"))
        )
    except:
        raise RuntimeError("No download links appeared in time.")

    # Find all data zip links
    zip_links = self.driver.find_elements(
        By.XPATH, "//a[@data-download-link='true' and contains(text(), 'Zip File')]"
    )

    if not zip_links:
        raise RuntimeError("Zero Zip File links found. Refreshing...")

    # Debug print texts
    texts = [link.text.strip() for link in zip_links]
    print(f"Found {len(zip_links)} zip link(s) with labels: {texts}")

    # Click valid zip links (skip empty ones)
    clicked_count = 0
    for i, link in enumerate(zip_links):
        label = link.text.strip()
        if label == "":
            print(f"  Skipping link {i+1} (empty label)...")
            continue

        try:
            self.driver.execute_script('arguments[0].scrollIntoView(true);', link)
            time.sleep(0.5)
            self.driver.execute_script('arguments[0].click();', link)
            print(f'  Triggered download for: {label}...')
            clicked_count += 1
            time.sleep(3)  # inter-download pause
        except Exception as e:
            print(f'Error clicking zip link {i+1}: {e}')
            raise

    if clicked_count == 0:
        raise RuntimeError("Found links but none had valid labels. Refreshing...")

    # Finally, click the metadata link
    try:
        metadata_link = WebDriverWait(self.driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'Metadata')]"))
        )
        metadata_link.click()
        print('  Metadata download triggered.')
    except Exception as e:
        print(f'Metadata link click failed: {e}. Attempting fallback...')
        self.html.click_button('Metadata', By.PARTIAL_LINK_TEXT)

# Update the class method
ppmi_downloader.PPMIDownloader.download_all_zip_files = download_all_zip_files
print('Refined download_all_zip_files method (skips empty links) applied.')

Refined download_all_zip_files method (skips empty links) applied.


In [ ]:
def download_imaging_data_optimized(self, cohort: pd.DataFrame, timeout: float = 1200, destination_dir: str = "."):
    if len(cohort) == 0: return
    subjectIds = ",".join(cohort["PATNO"].astype(str).unique())
    collection_name = str(cohort["PATNO"].iloc[0])

    # 1. Login and Search
    print("Logging in and navigating to search...")
    self.driver.get(ppmi_downloader.ppmi_navigator.ppmi_main_webpage)
    self.html.login(self.email, self.password)
    self.driver.get(ppmi_downloader.ppmi_navigator.ppmi_query_page)

    WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.ID, "subjectIdText"))).send_keys(subjectIds)
    self.driver.execute_script("arguments[0].click();", self.driver.find_element(By.ID, "advSearchQuery"))

    # 2. Add to Collection
    self.html.Search_AdvancedImageSearchbeta_SelectAll()
    self.driver.execute_script("arguments[0].click();", WebDriverWait(self.driver, 10).until(EC.element_to_be_clickable((By.ID, "advResultAddCollectId"))))
    WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.ID, "nameText"))).send_keys(collection_name)
    self.html.Search_AdvancedImageSearchbeta_AddToCollection_OK()

    # 3. Navigate to Export Tab
    print(f"Navigating to Export page for {collection_name}...")
    self.driver.execute_script("arguments[0].click();", WebDriverWait(self.driver, 10).until(EC.element_to_be_clickable((By.ID, "export"))))

    # Increased to 10 attempts for higher reliability
    max_download_attempts = 10
    for attempt in range(max_download_attempts):
        print(f"\n--- Attempt {attempt + 1} ---")
        time.sleep(3)   # ensures the export tab is properly loaded before we start clicking

        try:
            # 4. Selection
            self.scroll_and_select(cohort)
            self.driver.save_screenshot(f"debug_1_selected_attempt_{attempt+1}.png")

            # 5. Open Download Overlay
            print("Initiating download overlay...")
            dl_btn = self.driver.find_element(By.ID, "simple-download-button")
            self.driver.execute_script("arguments[0].click();", dl_btn)
            time.sleep(10)
            self.driver.save_screenshot(f"debug_2_overlay_attempt_{attempt+1}.png")

            # 6. Download Links
            self.download_all_zip_files()
            break # Success

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}. Refreshing page and retrying...")
            self.driver.refresh()
    else:
        raise RuntimeError("Failed to complete download after 10 attempts.")

    # 7. Wait for completion and Unzip
    time.sleep(1)   # important fix to ensure the metadata download starts before the download_complete condition is checked
    def download_complete(driver):
        files = os.listdir(self.tempdir.name)
        return len(files) > 0 and not any(f.endswith(".crdownload") for f in files)

    print("Waiting for files to finish downloading...")
    WebDriverWait(self.driver, timeout).until(download_complete)
    self.html.unzip_imaging_data(os.listdir(self.tempdir.name), self.tempdir.name, destination_dir)
    print("Download and extraction complete.")

# Apply the optimized version to the class
ppmi_downloader.PPMIDownloader.download_imaging_data = download_imaging_data_optimized
print("Concise download_imaging_data_optimized updated with 10 retry attempts.")

Concise download_imaging_data_optimized updated with 10 retry attempts.


In [ ]:
import re

def prepare_patient_cohort(selected_patno):
    """
    Filters df_metadata for a specific patient and prepares the cohort format for the downloader.
    Includes Image ID to ensure unique scan selection.
    """
    df_subject_raw = df_metadata[df_metadata['Subject ID'].astype(str) == str(selected_patno)].copy()

    if df_subject_raw.empty:
        print(f"Warning: No metadata found for Subject {selected_patno}")
        return pd.DataFrame()

    def extract_patno(val):
        match = re.search(r'(\d+)', str(val))
        return int(match.group(1)) if match else None

    df_subject_raw['PATNO'] = df_subject_raw['Subject ID'].apply(extract_patno)

    visit_col = 'Visit_Short' if 'Visit_Short' in df_subject_raw.columns else 'Visit'
    df_subject_raw['EVENT_ID'] = df_subject_raw[visit_col]

    # Include Image ID in the selection to distinguish between identical descriptions
    df_selected_cohort = df_subject_raw[['PATNO', 'EVENT_ID', 'Description', 'Image ID']].drop_duplicates()

    print(f"Prepared mini-cohort for Subject {selected_patno} with {len(df_selected_cohort)} records (using unique Image IDs).")
    return df_selected_cohort

print("prepare_patient_cohort updated to include Image ID.")

prepare_patient_cohort updated to include Image ID.


# Fastsurfer Setup

FastSurfer requires NIfTI (.nii, .nii.gz) or MGH (.mgz) files. PPMI downloads MRI in DICOM format. There seems to have been an option in the past to download in NIfTI but it's no longer available.

In [ ]:
#@title Install dcm2niix for DICOM conversion
!apt-get update -qq && apt-get install -y dcm2niix

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libturbojpeg libyaml-cpp0.7
The following NEW packages will be installed:
  dcm2niix libturbojpeg libyaml-cpp0.7
0 upgraded, 3 newly installed, 0 to remove and 60 not upgraded.
Need to get 529 kB of archives.
After this operation, 1,877 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libturbojpeg amd64 2.1.2-0ubuntu1 [175 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libyaml-cpp0.7 amd64 0.7.0+dfsg-8build1 [97.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 dcm2niix amd64 1.0.20211006-1build1 [256 kB]
Fetched 529 kB in 1s (438 kB/s)
Selecting previously unselected pa

In [ ]:
#@title Setup the environment by downloading the stable branch of deep-mi/fastsurfer project and the required packages

import os
import sys
from os.path import exists, join, basename, splitext

print("Starting setup. This could take a few minutes")
print("----------------------------------------------")

is_google_colab = "colab.research.google.com" in str(os.environ)
if is_google_colab:
    # this is for a Google Colab Notebook
    SETUP_DIR = "/content/"
else:
    SETUP_DIR = os.environ["HOME"] + "/fastsurfer_files/"

# Go to the FastSurfer directory
!mkdir -p "{SETUP_DIR}"
%cd "{SETUP_DIR}"

print(f"Using {SETUP_DIR} to store files.")

print("Downloading FastSurfer")
print("----------------------------------------------")


git_repo_url = 'https://github.com/deep-mi/fastsurfer.git'
project_name = splitext(basename(git_repo_url))[0]
FASTSURFER_HOME = SETUP_DIR + project_name + "/"
if not exists(project_name):
  # clone and install dependencies
  ! git clone -q --branch stable $git_repo_url
  ! pip install -r $FASTSURFER_HOME/requirements.txt
sys.path.append(FASTSURFER_HOME)

# Update dependencies
print("Installing required packages")
print("----------------------------------------------")

! pip install torchio==0.18.83
! pip install yacs==0.1.8
! pip install plotly==5.9.0

print("Finished setup")
print("----------------------------------------------")

Starting setup. This could take a few minutes
----------------------------------------------
/content
Using /content/ to store files.
----------------------------------------------
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.2/101.2 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 116.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 225.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 113.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 248.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 73.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 136.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 147.9 MB/s eta 0:00:00
     ━━━

In [ ]:
import subprocess

class FastSurferSegmentationError(Exception):
    """Custom exception for specific segmentation failures like division by zero."""
    pass

def run_fastsurfer_robust_v2(cmd):
    """
    Executes FastSurfer and monitors output.
    Terminates on success OR known fatal errors.
    """
    print(f"Executing command:\n{cmd}\n")
    print("--- Output Log (Robust v2.2) ---")

    process = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )

    error_detected = False
    for line in process.stdout:
        print(line, end='')

        # Success Check
        if "Processing whole pipeline finished" in line:
            print("\n>>> Success criteria met. Terminating process.")
            process.terminate()
            break

        # Check for multiple variations of the known stats/seg failures
        known_errors = [
            "ERROR: asegdkt statsfile generation failed!",
            "ERROR: FastSurfer asegdkt segmentation failed",
            "ERROR: Cerebellum Segmentation failed!",
            "ERROR: Biasfield correction failed!"
        ]

        if any(err in line for err in known_errors):
            print(f"\n>>> KNOWN ERROR DETECTED: {line.strip()}. Terminating process to skip.")
            error_detected = True
            process.terminate()
            break

    process.wait()
    print("--- Process Finished ---")

    if error_detected:
        raise FastSurferSegmentationError("Segmentation/Stats/Biasfield failure detected")

In [ ]:
import glob
import os

def construct_command(selected_patno):
    """
    Constructs a list of FastSurfer cross-sectional commands, one for each MRI session.
    Args:
        selected_patno (str): The ID of the patient to process.
    Returns:
        list: A list of constructed shell commands.
    """
    # 1. Define where we expect the NIfTI files to be located
    input_dir = f"/content/data/{selected_patno}"
    nifti_files = sorted(glob.glob(os.path.join(input_dir, "*.nii.gz")))

    if not nifti_files:
        print(f"No NIfTI files found for patient {selected_patno} in {input_dir}")
        return []

    commands = []

    # 2. Loop over each image to create a cross-sectional command
    for img in nifti_files:
        # We use the filename (without extension) as part of the --sid to distinguish sessions
        session_id = f"{selected_patno}_{os.path.basename(img).replace('.nii.gz', '')}"

        cmd = (
            f"export FASTSURFER_HOME={FASTSURFER_HOME}; "
            f"{FASTSURFER_HOME}/run_fastsurfer.sh "
            f"--t1 {img} "
            f"--sd {SETUP_DIR}fastsurfer_seg "
            f"--sid {session_id} "
            f"--seg_only "
            f"--py python3 "
            f"--3T "
            f"--allow_root"
        )
        commands.append(cmd)

    return commands

print("construct_command updated to return a list of session-specific commands.")

construct_command updated to return a list of session-specific commands.


In [ ]:
import shutil
import os
import glob

def zip_and_save_stats(selected_patno):
    """
    Zips all stats directories for a specific patient's cross-sectional sessions and saves to Drive.
    Args:
        selected_patno (str): The ID of the patient.
    """
    output_base_dir = f"{SETUP_DIR}fastsurfer_seg"
    drive_export_dir = "/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/"
    zip_filename = f"stats_{selected_patno}.zip"
    zip_path = os.path.join("/content", zip_filename)

    if not os.path.exists(output_base_dir):
        print(f"Output directory {output_base_dir} does not exist.")
        return

    # 1. Identify all directories belonging to this subject
    # In the cross-sectional setup, we named sids as {selected_patno}_{session}
    all_dirs = [d for d in os.listdir(output_base_dir) if os.path.isdir(os.path.join(output_base_dir, d))]

    # Filter for directories starting with the patient ID
    subject_dirs = [d for d in all_dirs if d.startswith(str(selected_patno))]

    if not subject_dirs:
        print(f"No stats directories found for patient {selected_patno} in {output_base_dir}.")
        return

    # 2. Create a temporary staging area
    temp_zip_dir = f"/content/temp_stats_{selected_patno}"
    if os.path.exists(temp_zip_dir):
        shutil.rmtree(temp_zip_dir)
    os.makedirs(temp_zip_dir, exist_ok=True)

    print(f"Collecting stats from sessions: {subject_dirs}")
    found_any_stats = False
    for d in subject_dirs:
        src_stats = os.path.join(output_base_dir, d, "stats")
        if os.path.exists(src_stats):
            dest = os.path.join(temp_zip_dir, d)
            shutil.copytree(src_stats, dest)
            found_any_stats = True
        else:
            print(f"  Warning: No stats folder found in {d}")

    if not found_any_stats:
        print(f"No actual 'stats' folders found to zip for {selected_patno}.")
        return

    # 3. Zip and Move
    print(f"Zipping collected session stats for patient {selected_patno}...")
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', temp_zip_dir)

    os.makedirs(drive_export_dir, exist_ok=True)
    final_drive_path = os.path.join(drive_export_dir, zip_filename)

    if os.path.exists(final_drive_path):
        os.remove(final_drive_path)

    shutil.move(zip_path, final_drive_path)

    # Clean up
    shutil.rmtree(temp_zip_dir)
    print(f"Stats successfully saved to Drive: {final_drive_path}")

print('zip_and_save_stats updated for cross-sectional session collection.')

zip_and_save_stats updated for cross-sectional session collection.


# Processing Loop

## Main Loop

In [ ]:
'''

import datetime
import os

# Main Processing Loop
# Iterates through subjects marked as 'pending' or 'failed'
for index, row in df_checkpoint[df_checkpoint['status'].isin(['pending', 'failed'])].iterrows():
    subject_id = str(row['Subject ID'])
    print(f"\n{'='*80}\nSTARTING PROCESS FOR SUBJECT: {subject_id}\n{'='*80}")

    try:
        # 1. Initialize PPMI Downloader
        print(f"[STEP 1] Initializing PPMIDownloader for {subject_id}...")
        ppmi = ppmi_downloader.PPMIDownloader(config_file=CONFIG_FILE, headless=True)

        # 2. Prepare Cohort Metadata
        print(f"[STEP 2] Preparing cohort metadata for {subject_id}...")
        cohort = prepare_patient_cohort(subject_id)
        if cohort.empty:
            print(f"!!! Skipping {subject_id}: Metadata filtering returned empty result.")
            df_checkpoint.at[index, 'status'] = 'failed'
            continue

        # 3. Setup Local Workspace
        subject_dir = f"/content/data/{subject_id}"
        print(f"[STEP 3] Setting up directory: {subject_dir}")
        if os.path.exists(subject_dir):
            shutil.rmtree(subject_dir)
        os.makedirs(subject_dir, exist_ok=True)

        # 4. Download Imaging Data
        print(f"[STEP 4] Downloading DICOM scans from PPMI for {subject_id}...")
        ppmi.download_imaging_data(cohort, destination_dir=subject_dir)

        # 5. DICOM to NIfTI Conversion
        print(f"[STEP 5] Converting DICOMs to NIfTI using dcm2niix...")
        conv_cmd = f"dcm2niix -z y -f %p_%t -o {subject_dir} {subject_dir}"
        os.system(conv_cmd)

        # 6. Construct FastSurfer Commands (returns a list for cross-sectional)
        print(f"[STEP 6] Constructing session-specific FastSurfer commands...")
        fs_commands = construct_command(subject_id)

        if not fs_commands:
            raise RuntimeError(f"No commands generated for {subject_id}. Check if NIfTI files exist in {subject_dir}")

        # 7. Execute Segmentation for each session
        print(f"[STEP 7] Launching Segmentation Pipeline for {len(fs_commands)} sessions...")
        for i, cmd in enumerate(fs_commands):
            print(f"\n--- Processing Session {i+1}/{len(fs_commands)} ---")
            # We use the cross-sectional robust wrapper
            run_fastsurfer_robust(cmd)

        # 8. Check Outputs and Save Results (Zips all sessions found)
        print(f"[STEP 8] Verifying outputs and zipping all session stats...")
        zip_and_save_stats(subject_id)

        # 9. Update Checkpoint on Success
        df_checkpoint.at[index, 'status'] = 'completed'
        df_checkpoint.at[index, 'timestamp'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\nSUCCESS: Subject {subject_id} processed completely ({len(fs_commands)} sessions).")

    except Exception as e:
        print(f"\n!!! CRITICAL ERROR for Subject {subject_id}: {e}")
        df_checkpoint.at[index, 'status'] = 'failed'
        df_checkpoint.at[index, 'timestamp'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    finally:
        # Clean up Driver session
        try:
            ppmi.driver.quit()
            print("[INFO] Selenium driver closed.")
        except:
            pass

        # Save checkpoint progress to Drive after every subject
        df_checkpoint.to_csv(checkpoint_path, index=False)
        print(f"[INFO] Checkpoint saved to Drive.")

print("\nALL SUBJECTS IN QUEUE PROCESSED.")

'''

'\n\nimport datetime\nimport os\n\n# Main Processing Loop\n# Iterates through subjects marked as \'pending\' or \'failed\'\nfor index, row in df_checkpoint[df_checkpoint[\'status\'].isin([\'pending\', \'failed\'])].iterrows():\n    subject_id = str(row[\'Subject ID\'])\n    print(f"\n{\'=\'*80}\nSTARTING PROCESS FOR SUBJECT: {subject_id}\n{\'=\'*80}")\n\n    try:\n        # 1. Initialize PPMI Downloader\n        print(f"[STEP 1] Initializing PPMIDownloader for {subject_id}...")\n        ppmi = ppmi_downloader.PPMIDownloader(config_file=CONFIG_FILE, headless=True)\n\n        # 2. Prepare Cohort Metadata\n        print(f"[STEP 2] Preparing cohort metadata for {subject_id}...")\n        cohort = prepare_patient_cohort(subject_id)\n        if cohort.empty:\n            print(f"!!! Skipping {subject_id}: Metadata filtering returned empty result.")\n            df_checkpoint.at[index, \'status\'] = \'failed\'\n            continue\n\n        # 3. Setup Local Workspace\n        subject_dir = 

In [ ]:
import datetime
import os
import shutil

# Reversed Main Processing Loop (v2 with Session Tracking)
# Iterates from the BOTTOM up for subjects marked as 'pending' or 'failed'
for index, row in df_checkpoint[df_checkpoint['status'].isin(['pending', 'failed'])][::-1].iterrows():
    subject_id = str(row['Subject ID'])
    print(f"\n{'='*80}\nSTARTING PROCESS FOR SUBJECT (REVERSED): {subject_id}\n{'='*80}")

    try:
        # 1. Initialize PPMI Downloader
        print(f"[STEP 1] Initializing PPMIDownloader for {subject_id}...")
        ppmi = ppmi_downloader.PPMIDownloader(config_file=CONFIG_FILE, headless=True)

        # 2. Prepare Cohort Metadata
        print(f"[STEP 2] Preparing cohort metadata for {subject_id}...")
        cohort = prepare_patient_cohort(subject_id)
        if cohort.empty:
            print(f"!!! Skipping {subject_id}: Metadata filtering returned empty result.")
            df_checkpoint.at[index, 'status'] = 'failed'
            continue

        # 3. Setup Local Workspace
        subject_dir = f"/content/data/{subject_id}"
        print(f"[STEP 3] Setting up directory: {subject_dir}")
        if os.path.exists(subject_dir):
            shutil.rmtree(subject_dir)
        os.makedirs(subject_dir, exist_ok=True)

        # 4. Download Imaging Data
        print(f"[STEP 4] Downloading DICOM scans from PPMI for {subject_id}...")
        ppmi.download_imaging_data(cohort, destination_dir=subject_dir)

        # 5. DICOM to NIfTI Conversion
        print(f"[STEP 5] Converting DICOMs to NIfTI using dcm2niix...")
        conv_cmd = f"dcm2niix -z y -f %p_%t -o {subject_dir} {subject_dir}"
        os.system(conv_cmd)

        # 6. Construct FastSurfer Commands
        print(f"[STEP 6] Constructing session-specific FastSurfer commands...")
        fs_commands = construct_command(subject_id)

        if not fs_commands:
            raise RuntimeError(f"No commands generated for {subject_id}. Check if NIfTI files exist in {subject_dir}")

        # 7. Execute Segmentation for each session with Success Tracking
        print(f"[STEP 7] Launching Segmentation Pipeline for {len(fs_commands)} sessions...")
        successful_sessions = 0
        for i, cmd in enumerate(fs_commands):
            print(f"\n--- Processing Session {i+1}/{len(fs_commands)} ---")
            try:
                run_fastsurfer_robust_v2(cmd)
                successful_sessions += 1
            except FastSurferSegmentationError as e:
                print(f"\n>>> Skipping session {i+1} due to known error: {e}")

        # 8. Check Outputs and Save Results
        print(f"[STEP 8] Verifying outputs and zipping all session stats...")
        zip_and_save_stats(subject_id)

        # 9. Update Checkpoint with Success Ratio
        status_text = 'completed' if successful_sessions == len(fs_commands) else f"{successful_sessions}/{len(fs_commands)} completed"
        df_checkpoint.at[index, 'status'] = status_text
        df_checkpoint.at[index, 'timestamp'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\nDONE: Subject {subject_id} finalized with status: {status_text}")

    except Exception as e:
        print(f"\n!!! CRITICAL ERROR for Subject {subject_id}: {e}")
        df_checkpoint.at[index, 'status'] = 'failed'
        df_checkpoint.at[index, 'timestamp'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    finally:
        # Clean up Driver session
        try:
            ppmi.driver.quit()
            print("[INFO] Selenium driver closed.")
        except:
            pass

        # Save checkpoint progress to Drive after every subject
        df_checkpoint.to_csv(checkpoint_path, index=False)
        print(f"[INFO] Checkpoint saved to Drive.")

print("\nALL SUBJECTS IN QUEUE PROCESSED (REVERSED).")


STARTING PROCESS FOR SUBJECT (REVERSED): 101038
[STEP 1] Initializing PPMIDownloader for 101038...
Using Chrome binary at: /usr/bin/chromium-browser


LivingPark-utils|DEBUG|ppmi_downloader.py:173 in __init__()
                       self.driver.capabilities: {'acceptInsecureCerts': False,
                                                  'browserName': 'chrome',
                                                  'browserVersion': '108.0.5359.40',
                                                  'chrome': {'chromedriverVersion': '108.0.5359.40 '
                                                                                    '(280b5fcaab3e877562b06cfaf2eb51121e13c3b9-refs/branch-heads/5359@{#689})',
                                                             'userDataDir': '/tmp/.org.chromium.Chromium.Fn2vqA'},
                                                  'goog:chromeOptions': {'debuggerAddress': 'localhost:39797'},
                                                  'networkConnectionEnabled': False,
                                                  'pageLoadStrategy': 'normal',
                                               

[STEP 2] Preparing cohort metadata for 101038...
Prepared mini-cohort for Subject 101038 with 2 records (using unique Image IDs).
[STEP 3] Setting up directory: /content/data/101038
[STEP 4] Downloading DICOM scans from PPMI for 101038...
Logging in and navigating to search...


LivingPark-utils|DEBUG|ppmi_navigator.py:221 in click_button()
                       "Click button": 'Click button'
                       field: 'ida-cookie-policy-accept'
                       debug_name: 'Cookie Policy'
                       caller: 'ppmi_navigator.py:332 validate_cookie_policy'
                       current_url: 'https://ida.loni.usc.edu/login.jsp?project=PPMI'
LivingPark-utils|DEBUG|ppmi_navigator.py:221 in click_button()
                       "Click button": 'Click button'
                       field: 'ida-cookie-policy-accept'
                       debug_name: 'Cookie Policy'
                       caller: 'ppmi_navigator.py:228 click_button'
                       current_url: 'https://ida.loni.usc.edu/login.jsp?project=PPMI'
LivingPark-utils|DEBUG|ppmi_navigator.py:124 in wait_for_element_to_be_visible()
                       "Wait for element to be visible": 'Wait for element to be visible'
                       field: 'ida-menu-option.sub-menu.user'

Navigating to Export page for 101038...

--- Attempt 1 ---
Selecting target scans via scrolling...
  Checked: ('101038', 'V04', '3D T1')
  Checked: ('101038', 'BL', '3D T1')
All target scans successfully selected.
Initiating download overlay...
Found 2 zip link(s) with labels: ['Zip File', 'Zip File']
  Triggered download for: Zip File...
  Triggered download for: Zip File...
  Metadata download triggered.
Waiting for files to finish downloading...


  0%|          | 0/4 [00:00<?, ?it/s]LivingPark-utils|INFO|f"Successfully downloaded files {filesname}...": 'Successfully downloaded files ['PPMI/101038/3D_T1/2022-06-09_16_58_22.0/I1616556/S1155943I1616556.xml', 'PPMI/101038/3D_T1/2021-04-27_13_12_13.0/I11081665/S5848090I11081665.xml']...'
LivingPark-utils|INFO|f"Successfully downloaded file {filename}": 'Successfully downloaded file blank.csv'
LivingPark-utils|INFO|f"Successfully downloaded files {filesname}...": 'Successfully downloaded files ['PPMI/101038/3D_T1/2022-06-09_16_58_22.0/I1616556/PPMI_101038_MR_3D_T1__br_raw_20220826134058361_50_S1155943_I1616556.dcm', 'PPMI/101038/3D_T1/2022-06-09_16_58_22.0/I1616556/PPMI_101038_MR_3D_T1__br_raw_20220826134102790_27_S1155943_I1616556.dcm']...'
100%|██████████| 4/4 [00:00<00:00, 10.70it/s]


Download and extraction complete.
[STEP 5] Converting DICOMs to NIfTI using dcm2niix...
[STEP 6] Constructing session-specific FastSurfer commands...
[STEP 7] Launching Segmentation Pipeline for 2 sessions...

--- Processing Session 1/2 ---
Executing command:
export FASTSURFER_HOME=/content/fastsurfer/; /content/fastsurfer//run_fastsurfer.sh --t1 /content/data/101038/Anon_20210427130053.nii.gz --sd /content/fastsurfer_seg --sid 101038_Anon_20210427130053 --seg_only --py python3 --3T --allow_root

--- Output Log (Robust v2.2) ---
time command failing, not using time...
  running FastSurfer as root, because it will lead to files and folders created as root.
  If you are running FastSurfer in a docker container, you can specify the user
  with '-u $(id -u):$(id -g)' (see https://docs.docker.com/engine/reference/run/#user).
Start of the log for a new run_fastsurfer.sh invocation
Version: 2.4.2+7e53343
Sun Apr  5 10:35:48 PM UTC 2026

Log file for FastSurfer pipeline, run_fastsurfer.sh and 

## Processing of manual downloads for bugged entries

In [ ]:
import os
import shutil
import zipfile
import glob

# --- Configuration ---
subject_id = "3866"
working_dir = f"/content/data/{subject_id}"
zip_scan = "/content/3866.zip"
zip_meta = "/content/3866_IDA_Metadata.zip"

# 1. Unpack uploaded files
print(f"--- Unpacking data for {subject_id} ---")
if os.path.exists(working_dir): shutil.rmtree(working_dir)
os.makedirs(working_dir, exist_ok=True)

for z in [zip_scan, zip_meta]:
    if os.path.exists(z):
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall(working_dir)
        print(f"  Unzipped: {z}")
    else:
        print(f"  Warning: {z} not found in /content/")

# 2. Convert DICOM to NIfTI
print("--- Converting DICOMs ---")
os.system(f"dcm2niix -z y -f %p_%t -o {working_dir} {working_dir}")

# 3. Construct and Run FastSurfer
print("--- Running FastSurfer ---")
# We assume the conversion produced at least one .nii.gz file
nifti_files = sorted(glob.glob(os.path.join(working_dir, "*.nii.gz")))

if not nifti_files:
    print("Error: No NIfTI files found after conversion.")
else:
    # Process the first valid NIfTI found
    img = nifti_files[0]
    session_id = f"{subject_id}_manual_fix"
    fs_cmd = (
        f"export FASTSURFER_HOME={FASTSURFER_HOME}; "
        f"{FASTSURFER_HOME}/run_fastsurfer.sh "
        f"--t1 {img} --sd {SETUP_DIR}fastsurfer_seg --sid {session_id} "
        f"--seg_only --py python3 --3T --allow_root"
    )

    try:
        # Using the robust v2 wrapper to catch errors
        run_fastsurfer_robust_v2(fs_cmd)

        # 4. Custom Merging Logic to Drive
        print("--- Merging new stats into Drive Zip ---")
        drive_zip = f"/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/stats_{subject_id}.zip"
        staging_dir = f"/content/merge_staging_{subject_id}"
        if os.path.exists(staging_dir): shutil.rmtree(staging_dir)
        os.makedirs(staging_dir, exist_ok=True)

        # Extract old if exists
        if os.path.exists(drive_zip):
            with zipfile.ZipFile(drive_zip, 'r') as zr: zr.extractall(staging_dir)
            print("  Extracted existing zip from Drive.")

        # Copy new stats
        new_stats_path = os.path.join(SETUP_DIR, "fastsurfer_seg", session_id, "stats")
        if os.path.exists(new_stats_path):
            shutil.copytree(new_stats_path, os.path.join(staging_dir, session_id))

            # Re-zip
            shutil.make_archive(f"/content/stats_{subject_id}", 'zip', staging_dir)
            shutil.move(f"/content/stats_{subject_id}.zip", drive_zip)
            print(f"SUCCESS: Combined stats saved to {drive_zip}")
        else:
            print("Error: New stats folder not found. Check FastSurfer logs.")

    except Exception as e:
        print(f"Processing failed: {e}")

--- Unpacking data for 3866 ---
  Unzipped: /content/3866.zip
  Unzipped: /content/3866_IDA_Metadata.zip
--- Converting DICOMs ---
--- Running FastSurfer ---
Executing command:
export FASTSURFER_HOME=/content/fastsurfer/; /content/fastsurfer//run_fastsurfer.sh --t1 /content/data/3866/MPRAGE_GRAPPA_20140926092006.nii.gz --sd /content/fastsurfer_seg --sid 3866_manual_fix --seg_only --py python3 --3T --allow_root

--- Output Log (Robust v2.2) ---
time command failing, not using time...
  running FastSurfer as root, because it will lead to files and folders created as root.
  If you are running FastSurfer in a docker container, you can specify the user
  with '-u $(id -u):$(id -g)' (see https://docs.docker.com/engine/reference/run/#user).
Start of the log for a new run_fastsurfer.sh invocation
Version: 2.4.2+7e53343
Sun Apr  5 10:47:54 PM UTC 2026

Log file for FastSurfer pipeline, run_fastsurfer.sh and segmentation(s)
python3 /content/fastsurfer//FastSurferCNN/run_prediction.py --t1 /cont

In [ ]:
import os
import shutil
import zipfile
import datetime

subject_id = "3866"
old_session_id = "3866_manual_fix"
new_session_id = f"{subject_id}_MPRAGE_GRAPPA_2014"

# 1. Rename the existing processed folder
old_path = os.path.join(SETUP_DIR, "fastsurfer_seg", old_session_id)
new_path = os.path.join(SETUP_DIR, "fastsurfer_seg", new_session_id)

if os.path.exists(old_path):
    if os.path.exists(new_path): shutil.rmtree(new_path)
    os.rename(old_path, new_path)
    print(f"Renamed folder: {old_session_id} -> {new_session_id}")

# 2. Merge into Drive Zip
drive_zip = f"/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/stats_{subject_id}.zip"
staging_dir = f"/content/merge_staging_{subject_id}"
if os.path.exists(staging_dir): shutil.rmtree(staging_dir)
os.makedirs(staging_dir, exist_ok=True)

# Extract existing data if it exists
if os.path.exists(drive_zip):
    with zipfile.ZipFile(drive_zip, 'r') as zr:
        zr.extractall(staging_dir)
    print("Extracted existing zip from Drive.")

# Copy the new stats (with the new folder name)
new_stats_src = os.path.join(new_path, "stats")
if os.path.exists(new_stats_src):
    dest = os.path.join(staging_dir, new_session_id)
    if os.path.exists(dest): shutil.rmtree(dest)
    shutil.copytree(new_stats_src, dest)

    # Re-zip and update Drive
    shutil.make_archive(f"/content/stats_{subject_id}", 'zip', staging_dir)
    shutil.move(f"/content/stats_{subject_id}.zip", drive_zip)
    print(f"SUCCESS: Merged stats saved to {drive_zip}")

    # 3. Update Checkpoint to completed
    idx = df_checkpoint[df_checkpoint['Subject ID'].astype(str) == subject_id].index
    if not idx.empty:
        df_checkpoint.at[idx[0], 'status'] = 'completed'
        df_checkpoint.at[idx[0], 'timestamp'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        df_checkpoint.to_csv(checkpoint_path, index=False)
        print(f"Subject {subject_id} marked as completed in checkpoint.")
else:
    print(f"Error: Stats folder not found at {new_stats_src}")

Renamed folder: 3866_manual_fix -> 3866_MPRAGE_GRAPPA_2014
Extracted existing zip from Drive.
SUCCESS: Merged stats saved to /content/drive/MyDrive/Fastsurfer_Automated_Pipeline/stats_3866.zip
Subject 3866 marked as completed in checkpoint.


In [ ]:
import zipfile
import os
import shutil

subject_id = '3866'
drive_zip = f'/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/stats_{subject_id}.zip'
staging_dir = f'/content/final_cleanup_{subject_id}'

if os.path.exists(drive_zip):
    # 1. Extract to a fresh staging area
    if os.path.exists(staging_dir): shutil.rmtree(staging_dir)
    os.makedirs(staging_dir)

    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(staging_dir)

    # 2. Identify and remove the 'manual_fix' folder
    # It might be at the top level or nested depending on how it was zipped
    removed = False
    for root, dirs, files in os.walk(staging_dir):
        if '3866_manual_fix' in dirs:
            target = os.path.join(root, '3866_manual_fix')
            print(f'Removing redundant folder: {target}')
            shutil.rmtree(target)
            removed = True

    if removed:
        # 3. Re-zip the cleaned directory
        os.remove(drive_zip)
        shutil.make_archive(drive_zip.replace('.zip', ''), 'zip', staging_dir)
        print(f'SUCCESS: Cleaned {drive_zip}. "manual_fix" has been removed.')
    else:
        print('No "manual_fix" folder found in the ZIP. It appears already clean.')

    # Cleanup staging
    shutil.rmtree(staging_dir)
else:
    print(f'Error: {drive_zip} not found.')

Removing redundant folder: /content/final_cleanup_3866/3866_manual_fix
SUCCESS: Cleaned /content/drive/MyDrive/Fastsurfer_Automated_Pipeline/stats_3866.zip. "manual_fix" has been removed.


In [ ]:
import os
import glob

# List all ZIP files in the /content/ directory
zip_files = glob.glob('/content/*.zip')

print(f'--- ZIP Files found in /content/ ({len(zip_files)}) ---')
for f in sorted(zip_files):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f' - {os.path.basename(f)} ({size_mb:.2f} MB)')

if not zip_files:
    print('No ZIP files found. Please upload the subject data to /content/.')

--- ZIP Files found in /content/ (18) ---
 - 168082.zip (15.07 MB)
 - 168082_IDA_Metadata (1).zip (0.00 MB)
 - 168082_IDA_Metadata.zip (0.00 MB)
 - 168082_dataset.zip (15.19 MB)
 - 170727.zip (14.02 MB)
 - 170727_IDA_Metadata (1).zip (0.00 MB)
 - 170727_IDA_Metadata.zip (0.00 MB)
 - 170727_dataset.zip (13.71 MB)
 - 173266.zip (15.21 MB)
 - 173266_IDA_Metadata (1).zip (0.00 MB)
 - 173266_IDA_Metadata.zip (0.00 MB)
 - 173266_dataset.zip (16.13 MB)
 - 174131.zip (17.26 MB)
 - 174131_IDA_Metadata (1).zip (0.00 MB)
 - 174131_IDA_Metadata.zip (0.00 MB)
 - 174131_dataset.zip (17.14 MB)
 - 3866.zip (10.25 MB)
 - 3866_IDA_Metadata.zip (0.00 MB)


In [ ]:
import pandas as pd

target_subjects = ['168082', '170727', '173266', '174131']

print('--- Audit: Target Subjects Status ---')
display(df_checkpoint[df_checkpoint['Subject ID'].astype(str).isin(target_subjects)])

print('\n--- Audit: Summary of All Statuses ---')
status_summary = df_checkpoint['status'].value_counts()
print(status_summary)

# Check if anything else is non-completed
non_completed = df_checkpoint[~df_checkpoint['status'].str.contains('completed|download bug', case=False, na=False)]

if not non_completed.empty:
    print('\nWARNING: Found subjects that are neither completed nor marked as download bug:')
    display(non_completed)
else:
    print('\nSUCCESS: All other subjects are either completed or in the manual processing queue.')

--- Audit: Target Subjects Status ---


,Subject ID,status,timestamp,nipoppy BL
164,168082,download bug,2026-04-03 11:33:16,Yes
169,170727,download bug,2026-04-03 01:28:45,Yes
174,173266,download bug,2026-04-03 01:05:03,Yes
177,174131,download bug,2026-04-02 13:03:44,Yes



--- Audit: Summary of All Statuses ---
status
completed       378
download bug      4
Name: count, dtype: int64

SUCCESS: All other subjects are either completed or in the manual processing queue.


In [ ]:
import os
import shutil
import zipfile
import glob
import datetime

target_subjects = ['168082', '170727', '173266', '174131']

for subject_id in target_subjects:
    print(f"\n{'='*80}\nPROCESSING SUBJECT: {subject_id}\n{'='*80}")

    working_dir = f"/content/data/{subject_id}"
    if os.path.exists(working_dir): shutil.rmtree(working_dir)
    os.makedirs(working_dir, exist_ok=True)

    # 1. Identify and Unpack all ZIPs related to this subject
    subject_zips = glob.glob(f"/content/{subject_id}*.zip")
    print(f"Found {len(subject_zips)} ZIP files for extraction.")

    for z in subject_zips:
        print(f"  Extracting: {os.path.basename(z)}...")
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall(working_dir)

    # 2. DICOM to NIfTI Conversion
    print("Converting DICOMs to NIfTI...")
    os.system(f"dcm2niix -z y -f %p_%t -o {working_dir} {working_dir}")

    # 3. Construct and Run FastSurfer commands
    fs_commands = construct_command(subject_id)

    if not fs_commands:
        print(f"!!! Error: No NIfTI files generated for {subject_id}. Skipping.")
        continue

    print(f"Launching segmentation for {len(fs_commands)} sessions...")
    successful_sessions = 0
    for i, fs_cmd in enumerate(fs_commands):
        print(f"\n--- Session {i+1}/{len(fs_commands)} ---")
        try:
            # Using the robust v2 wrapper to catch known errors
            run_fastsurfer_robust_v2(fs_cmd)
            successful_sessions += 1
        except Exception as e:
            print(f">>> Session failed: {e}")

    # 4. Save Stats to Drive
    print("Zipping and saving stats to Drive...")
    zip_and_save_stats(subject_id)

    # 5. Update Checkpoint
    idx = df_checkpoint[df_checkpoint['Subject ID'].astype(str) == subject_id].index
    if not idx.empty:
        status_text = 'completed' if successful_sessions == len(fs_commands) else f"{successful_sessions}/{len(fs_commands)} completed"
        df_checkpoint.at[idx[0], 'status'] = status_text
        df_checkpoint.at[idx[0], 'timestamp'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        df_checkpoint.to_csv(checkpoint_path, index=False)
        print(f"SUCCESS: Subject {subject_id} finalized with status: {status_text}")

print("\nALL MANUAL SUBJECTS PROCESSED.")


PROCESSING SUBJECT: 168082
Found 4 ZIP files for extraction.
  Extracting: 168082.zip...
  Extracting: 168082_IDA_Metadata.zip...
  Extracting: 168082_dataset.zip...
  Extracting: 168082_IDA_Metadata (1).zip...
Converting DICOMs to NIfTI...
Launching segmentation for 2 sessions...

--- Session 1/2 ---
Executing command:
export FASTSURFER_HOME=/content/fastsurfer/; /content/fastsurfer//run_fastsurfer.sh --t1 /content/data/168082/Anon_20220916114239.nii.gz --sd /content/fastsurfer_seg --sid 168082_Anon_20220916114239 --seg_only --py python3 --3T --allow_root

--- Output Log (Robust v2.2) ---
time command failing, not using time...
  running FastSurfer as root, because it will lead to files and folders created as root.
  If you are running FastSurfer in a docker container, you can specify the user
  with '-u $(id -u):$(id -g)' (see https://docs.docker.com/engine/reference/run/#user).
Start of the log for a new run_fastsurfer.sh invocation
Version: 2.4.2+7e53343
Sun Apr  5 11:23:59 PM UTC

# Error Checks and Parsing the Results!

## Subject ID Issue (will have to accept this due to time constraints)

In [ ]:
import pandas as pd

# 1. Load the broader metadata file
broader_metadata_path = '/content/T1_metadata_Final_Cleaned (9 missing).csv'
df_broader = pd.read_csv(broader_metadata_path)

# 2. Ensure IDs and Visit columns are strings for consistent comparison
df_metadata['Subject ID'] = df_metadata['Subject ID'].astype(str)
df_metadata['Visit'] = df_metadata['Visit'].astype(str)
df_broader['Subject ID'] = df_broader['Subject ID'].astype(str)
df_broader['Visit'] = df_broader['Visit'].astype(str)

# 3. Identify duplicate scans in the broader dataset for the same Subject/Visit
# We group by Subject ID and Visit and count unique Image IDs
scan_counts = df_broader.groupby(['Subject ID', 'Visit'])['Image ID'].nunique().reset_index()
multi_scan_entries = scan_counts[scan_counts['Image ID'] > 1]

# 4. Filter the current df_metadata to see which of our active scans have siblings in the broader data
audit_results = pd.merge(
    df_metadata[['Subject ID', 'Visit', 'Image ID', 'Description']],
    multi_scan_entries[['Subject ID', 'Visit']],
    on=['Subject ID', 'Visit'],
    how='inner'
)

if audit_results.empty:
    print("SUCCESS: No records in df_metadata share a Subject ID and Visit with other Image IDs in the broader dataset.")
else:
    print(f"Found {len(audit_results)} records in your metadata that share a Subject/Visit with multiple Image IDs in the source file.")
    print("This confirms that the downloader may struggle to differentiate these scans if it doesn't use the Image ID.")
    display(audit_results.sort_values(by=['Subject ID', 'Visit']))

Found 113 records in your metadata that share a Subject/Visit with multiple Image IDs in the source file.
This confirms that the downloader may struggle to differentiate these scans if it doesn't use the Image ID.


,Subject ID,Visit,Image ID,Description
59,100001,Baseline,1473172,SAG 3D MPRAGE
87,100018,Baseline,1497578,3D T1-weighted
60,100891,Month 24,1677717,MPRAGE - Sag
61,101124,Baseline,1493057,3D T1-weighted
62,101799,Month 24,10253239,SAG 3D T1 FSPGR
...,...,...,...,...
55,4038,Month 24,497252,MPRAGE_GRAPPA_ADNI
56,4038,Month 48,901240,MPRAGE_GRAPPA
57,4081,Month 48,951750,MPRAGE GRAPPA
58,4082,Month 48,901269,MPRAGE GRAPPA


In [ ]:
import pandas as pd

# 1. Merge the active metadata with all records from the broader metadata for those Subject/Visit pairs
df_siblings = pd.merge(
    df_metadata[['Subject ID', 'Visit', 'Image ID']].rename(columns={'Image ID': 'Active_Image_ID'}),
    df_broader[['Subject ID', 'Visit', 'Image ID']].rename(columns={'Image ID': 'Broader_Image_ID'}),
    on=['Subject ID', 'Visit'],
    how='inner'
)

# 2. Filter for cases where the active scan has a sibling with a DIFFERENT Image ID
df_conflicts = df_siblings[df_siblings['Active_Image_ID'] != df_siblings['Broader_Image_ID']].copy()

# 3. Identify cases where our active scan has a HIGHER ID than at least one of its siblings
# This means a 'first-match' downloader would pick the wrong one.
df_conflicts['is_higher'] = df_conflicts['Active_Image_ID'] > df_conflicts['Broader_Image_ID']

# Group by the active scan to see if it's 'at risk'
risk_audit = df_conflicts.groupby(['Subject ID', 'Visit', 'Active_Image_ID'])['is_higher'].any().reset_index()

num_at_risk = risk_audit['is_higher'].sum()
total_scanned = len(risk_audit)

print(f"--- Download Risk Audit ---")
print(f"Total multi-scan Subject/Visit pairs checked: {total_scanned}")
print(f"Cases where your target Image ID is HIGHER than a duplicate: {num_at_risk}")

if num_at_risk > 0:
    print("\nThese subjects are highly likely to have had the wrong DICOMs downloaded if the script didn't match by ID:")
    display(risk_audit[risk_audit['is_higher'] == True].head(15))

--- Download Risk Audit ---
Total multi-scan Subject/Visit pairs checked: 113
Cases where your target Image ID is HIGHER than a duplicate: 107

These subjects are highly likely to have had the wrong DICOMs downloaded if the script didn't match by ID:


,Subject ID,Visit,Active_Image_ID,is_higher
1,100018,Baseline,1497578,True
2,100891,Month 24,1677717,True
3,101124,Baseline,1493057,True
4,101799,Month 24,10253239,True
5,102012,Baseline,1491294,True
6,102027,Baseline,1491306,True
7,102053,Baseline,1495921,True
8,102053,Month 12,1608787,True
9,102053,Month 24,10259812,True
11,102475,Baseline,1530293,True


In [ ]:
# Let's find the correct column names and compare Subject 100018
target_sub = '100018'
target_visit = 'Baseline'

# Identify siblings
siblings_detail = df_broader[(df_broader['Subject ID'] == target_sub) & (df_broader['Visit'] == target_visit)]

print(f"--- Metadata Comparison for Subject {target_sub} ({target_visit}) ---")

# Dynamically select columns that exist in the dataframe to avoid KeyErrors
all_cols = siblings_detail.columns.tolist()
useful_cols = ['Image ID', 'Description', 'Study Date', 'Manufacturer', 'Mfg Model', 'Slice Thickness', 'Field Strength']
available_cols = [c for c in useful_cols if c in all_cols]

display(siblings_detail[available_cols])

print("\nAvailable columns in df_broader:", all_cols)

--- Metadata Comparison for Subject 100018 (Baseline) ---


,Image ID,Description,Study Date,Manufacturer,Mfg Model,Slice Thickness,Field Strength
20,1497577,3D T1-weighted,2/02/2021,SIEMENS,Skyra,1.0,3.0
21,1497578,3D T1-weighted,2/02/2021,SIEMENS,Skyra,1.0,3.0



Available columns in df_broader: ['Subject ID', 'Sex', 'Visit', 'Study Date', 'Age', 'Description', 'Imaging Protocol', 'Image ID', 'Weighting', 'Acquisition Plane', 'Field Strength', 'Manufacturer', 'Mfg Model', 'Slice Thickness']


## Parsing

In [ ]:
import os
import zipfile
import io

# 1. Identify a valid stats ZIP on Drive
drive_path = '/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/'
zip_files = [f for f in os.listdir(drive_path) if f.startswith('stats_') and f.endswith('.zip')]

if not zip_files:
    print('No ZIP files found in the Drive folder.')
else:
    target_zip = os.path.join(drive_path, zip_files[0])
    print(f'Reading sample from: {os.path.basename(target_zip)}')

    # 2. Extract and display more of the first stats file found
    with zipfile.ZipFile(target_zip, 'r') as z:
        stats_entries = [f for f in z.namelist() if 'aseg+DKT.stats' in f]

        if not stats_entries:
            print('Could not find aseg+DKT.stats inside the ZIP.')
        else:
            sample_file = stats_entries[0]
            print(f'File found: {sample_file}\n')

            with z.open(sample_file) as f:
                content = f.read().decode('utf-8')
                lines = content.split('\n')
                # Increase to 300 lines to ensure we see the full structure table
                for line in lines[:300]:
                    print(line)

Reading sample from: stats_100001.zip
File found: 100001_Anon_20240911084930/aseg+DKT.stats

# Title Segmentation Statistics
#
# generating_program segstats.py
# FastSurfer_version 2.4.2
# cmdline /content/fastsurfer//FastSurferCNN/segstats.py --segfile /content/fastsurfer_seg/100001_Anon_20240911084930/mri/aparc.DKTatlas+aseg.deep.mgz --segstatsfile /content/fastsurfer_seg/100001_Anon_20240911084930/stats/aseg+DKT.stats --normfile /content/fastsurfer_seg/100001_Anon_20240911084930/mri/orig_nu.mgz --threads 1 --empty --excludeid 0 --sd /content/fastsurfer_seg --sid 100001_Anon_20240911084930 --ids 2 4 5 7 8 10 11 12 13 14 15 16 17 18 24 26 28 31 41 43 44 46 47 49 50 51 52 53 54 58 60 63 77 251 252 253 254 255 1002 1003 1005 1006 1007 1008 1009 1010 1011 1012 1013 1014 1015 1016 1017 1018 1019 1020 1021 1022 1023 1024 1025 1026 1027 1028 1029 1030 1031 1034 1035 2002 2003 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026 2027 2

In [ ]:
# 1. Parse the structure names directly from the loaded sample content
lines = content.split('\n')
structure_list = []
start_parsing = False

for line in lines:
    if line.startswith('# ColHeaders'):
        start_parsing = True
        continue
    # The table data starts after comments and header
    if start_parsing and line.strip() and not line.startswith('#'):
        parts = line.split()
        if len(parts) >= 5:
            # The 5th column (index 4) contains the Structure Name
            structure_list.append(parts[4])

print(f"--- Found {len(structure_list)} structures in sample file ---")
print("Full List for Template Construction:")
print(structure_list)

--- Found 100 structures in sample file ---
Full List for Template Construction:
['Left-Cerebral-White-Matter', 'Left-Lateral-Ventricle', 'Left-Inf-Lat-Vent', 'Left-Cerebellum-White-Matter', 'Left-Cerebellum-Cortex', 'Left-Thalamus', 'Left-Caudate', 'Left-Putamen', 'Left-Pallidum', '3rd-Ventricle', '4th-Ventricle', 'Brain-Stem', 'Left-Hippocampus', 'Left-Amygdala', 'CSF', 'Left-Accumbens-area', 'Left-VentralDC', 'Left-choroid-plexus', 'Right-Cerebral-White-Matter', 'Right-Lateral-Ventricle', 'Right-Inf-Lat-Vent', 'Right-Cerebellum-White-Matter', 'Right-Cerebellum-Cortex', 'Right-Thalamus', 'Right-Caudate', 'Right-Putamen', 'Right-Pallidum', 'Right-Hippocampus', 'Right-Amygdala', 'Right-Accumbens-area', 'Right-VentralDC', 'Right-choroid-plexus', 'WM-hypointensities', 'CC_Posterior', 'CC_Mid_Posterior', 'CC_Central', 'CC_Mid_Anterior', 'CC_Anterior', 'ctx-lh-caudalanteriorcingulate', 'ctx-lh-caudalmiddlefrontal', 'ctx-lh-cuneus', 'ctx-lh-entorhinal', 'ctx-lh-fusiform', 'ctx-lh-inferiorpa

In [ ]:
import pandas as pd

# 1. Define Identity Headings
identity_cols = ['PATNO', 'EVENT_ID', 'Study_Date', 'Image_Description']

# 2. Define 9 Global Measures
global_measures = [
    'MaskVol', 'BrainSegVol', 'BrainSegVolNotVent',
    'SupraTentorialVol', 'SupraTentorialVolNotVent',
    'SubCortGrayVol', 'rhCerebralWhiteMatterVol',
    'lhCerebralWhiteMatterVol', 'CerebralWhiteMatterVol'
]

# 3. Define 100 Anatomical Structures (from Left-Cerebral-White-Matter to ctx-rh-insula)
# These are extracted from the standard FreeSurfer/FastSurfer LUT used in the stats file
structures = structure_list

# 4. Create Empty DataFrame
all_columns = identity_cols + global_measures + structures
df_template = pd.DataFrame(columns=all_columns)

# 5. Save and Display
template_path = '/content/FastSurfer_Aggregation_Template.csv'
df_template.to_csv(template_path, index=False)

print(f"Empty spreadsheet created with {len(all_columns)} columns.")
print(f"Location: {template_path}")
display(df_template)

Empty spreadsheet created with 113 columns.
Location: /content/FastSurfer_Aggregation_Template.csv


,PATNO,EVENT_ID,Study_Date,Image_Description,MaskVol,BrainSegVol,BrainSegVolNotVent,SupraTentorialVol,SupraTentorialVolNotVent,SubCortGrayVol,...,ctx-rh-precentral,ctx-rh-precuneus,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-transversetemporal,ctx-rh-insula


In [ ]:
import re

# Search for eTIV in the sample stats content
print("--- Searching for eTIV / ICV ---")
etiv_pattern = re.compile(r'Measure (EstimatedTotalIntraCranialVol|eTIV|ICV),.*?, (\d+\.\d+), mm\^3', re.IGNORECASE)
etiv_match = etiv_pattern.search(content)

if etiv_match:
    print(f"Found eTIV measure: {etiv_match.group(1)} = {etiv_match.group(2)}")
else:
    print("EstimatedTotalIntraCranialVol not found in global measures.")
    # Peek at all lines starting with '# Measure'
    measure_lines = [l for l in content.split('\n') if l.startswith('# Measure')]
    print("Available Global Measures:")
    for ml in measure_lines: print(f"  {ml}")

--- Searching for eTIV / ICV ---
EstimatedTotalIntraCranialVol not found in global measures.
Available Global Measures:
  # Measure Mask, MaskVol, Mask Volume, 1446766.000000, mm^3
  # Measure BrainSeg, BrainSegVol, Brain Segmentation Volume, 1134900.497724, mm^3
  # Measure BrainSegNotVent, BrainSegVolNotVent, Brain Segmentation Volume Without Ventricles, 1107924.275239, mm^3
  # Measure SupraTentorial, SupraTentorialVol, Supratentorial volume, 991780.511670, mm^3
  # Measure SupraTentorialNotVent, SupraTentorialVolNotVent, Supratentorial volume, 964804.289185, mm^3
  # Measure SubCortGray, SubCortGrayVol, Subcortical gray matter volume, 58324.505092, mm^3
  # Measure rhCerebralWhiteMatter, rhCerebralWhiteMatterVol, Right hemisphere cerebral white matter volume, 243244.493056, mm^3
  # Measure lhCerebralWhiteMatter, lhCerebralWhiteMatterVol, Left hemisphere cerebral white matter volume, 245061.725970, mm^3
  # Measure CerebralWhiteMatter, CerebralWhiteMatterVol, Total cerebral white m

In [ ]:
import os
import zipfile
import pandas as pd
import re
from tqdm import tqdm

# --- Configuration ---
drive_results_dir = '/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/'
# Final output saved to the 'data' subfolder as requested
final_output_path = '/content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv'
os.makedirs('/content/drive/MyDrive/data/', exist_ok=True)

# 1. Get list of valid Subject IDs from df_metadata
valid_subjects = set(df_metadata['Subject ID'].astype(str).unique())

# Filter ZIP files to only include those in our metadata
all_zip_files = [f for f in os.listdir(drive_results_dir) if f.startswith('stats_') and f.endswith('.zip')]
zip_files = [f for f in all_zip_files if f.replace('stats_', '').replace('.zip', '') in valid_subjects]

print(f'Found {len(zip_files)} patient archives. Starting aggregation...')

all_rows = []

for zip_name in tqdm(zip_files):
    subject_id = zip_name.replace('stats_', '').replace('.zip', '')
    zip_path = os.path.join(drive_results_dir, zip_name)
    patient_meta = df_metadata[df_metadata['Subject ID'].astype(str) == subject_id].copy()

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            stats_files = [f for f in z.namelist() if f.endswith('aseg+DKT.stats')]
            for sf in stats_files:
                session_folder = sf.split('/')[0]
                date_match = re.search(r'20\d{6}', session_folder)
                folder_date_str = date_match.group(0) if date_match else ''

                best_match = None
                for _, meta_row in patient_meta.iterrows():
                    m_date = str(meta_row['Study Date'])
                    try:
                        formatted_meta_date = pd.to_datetime(m_date, format='%m-%d-%y').strftime('%Y%m%d')
                        if formatted_meta_date == folder_date_str:
                            best_match = meta_row
                            break
                    except:
                        continue

                if best_match is None and len(patient_meta) == 1:
                    best_match = patient_meta.iloc[0]

                if best_match is not None:
                    with z.open(sf) as f:
                        content = f.read().decode('utf-8')
                        lines = content.split('\n')
                        data_row = {'PATNO': subject_id,
                                    'EVENT_ID': best_match.get('Visit_Short', best_match.get('Visit')),
                                    'Study_Date': best_match['Study Date'],
                                    'Image_Description': best_match['Description']}

                        for gm in global_measures:
                            m_pattern = re.compile(rf'Measure\s+.*?\s+{gm},\s+.*?\s+([\d\.]+),\s+mm\^3', re.IGNORECASE)
                            m_match = m_pattern.search(content)
                            data_row[gm] = float(m_match.group(1)) if m_match else None

                        start_table = False
                        for line in lines:
                            if line.startswith('# ColHeaders'):
                                start_table = True
                                continue
                            if start_table and line.strip() and not line.startswith('#'):
                                parts = line.split()
                                if len(parts) >= 5 and parts[4] in structures:
                                    data_row[parts[4]] = float(parts[3])
                        all_rows.append(data_row)
    except Exception as e:
        print(f'Error processing {zip_name}: {e}')

# 2. Finalize master spreadsheet with Sorting
df_master = pd.DataFrame(all_rows, columns=all_columns)

# Sort by PATNO (alphabetical) and then by Study Date (chronological)
df_master['Study_Date_Dt'] = pd.to_datetime(df_master['Study_Date'], format='%m-%d-%y')
df_master = df_master.sort_values(by=['PATNO', 'Study_Date_Dt']).drop(columns=['Study_Date_Dt'])

df_master.to_csv(final_output_path, index=False)

print(f'\nAGGREGATION COMPLETE. Rows sorted and saved to: {final_output_path}')
display(df_master.head())

Found 382 patient archives. Starting aggregation...


100%|██████████| 382/382 [00:06<00:00, 59.86it/s]



AGGREGATION COMPLETE. Rows sorted and saved to: /content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv


,PATNO,EVENT_ID,Study_Date,Image_Description,MaskVol,BrainSegVol,BrainSegVolNotVent,SupraTentorialVol,SupraTentorialVolNotVent,SubCortGrayVol,...,ctx-rh-precentral,ctx-rh-precuneus,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-transversetemporal,ctx-rh-insula
1,100001,BL,10-07-20,SAG 3D MPRAGE,1426491.0,1.106751e+06,1.083366e+06,9.676464e+05,9.442612e+05,57988.070480,...,10456.012,8055.184,2290.155,9682.411,25368.864,7964.390,13764.618,7987.321,862.207,5872.390
2,100001,V06,11-29-22,SAG 3D MPRAGE,1408328.0,1.089433e+06,1.065102e+06,9.520664e+05,9.277358e+05,56786.189847,...,10350.497,8092.248,2238.521,9628.452,24891.809,7938.342,13677.405,7947.545,839.395,5757.502
0,100001,V10,09-11-24,SAG 3D T1 FSPGR,1446766.0,1.134900e+06,1.107924e+06,9.917805e+05,9.648043e+05,58324.505092,...,10653.921,7794.166,2495.664,10490.085,26054.235,7785.494,14174.507,8643.477,829.227,5873.501
1022,100005,BL,01-27-21,3D T1 _weighted,1707678.0,1.357007e+06,1.335605e+06,1.189561e+06,1.168159e+06,71864.305987,...,13115.193,11949.686,2701.991,11397.665,27442.435,12118.098,17765.782,11386.644,978.776,6785.774
1021,100005,V04,02-02-22,3D T1 _weighted,1703032.0,1.345458e+06,1.323268e+06,1.180172e+06,1.157982e+06,71555.081236,...,12558.124,11951.343,2637.227,11004.912,27114.055,11946.268,17658.720,10862.961,1003.884,6634.855


## Further error checks and edits

In [ ]:
# Error Check: Identify patients with identical structure measurements across multiple rows
print("--- Audit: Searching for Duplicate Measurement Rows ---")

# Group by PATNO and check for duplicates within the structure columns subset
structure_duplicates = df_master[df_master.duplicated(subset=['PATNO'] + structures, keep=False)]

if structure_duplicates.empty:
    print("SUCCESS: No patients have duplicate rows with identical anatomical measurements.")
else:
    print(f"ALERT: Found {len(structure_duplicates)} rows where measurements are identical for the same patient.")
    display(structure_duplicates[['PATNO', 'EVENT_ID', 'Study_Date'] + structures[:5]])

--- Audit: Searching for Duplicate Measurement Rows ---
SUCCESS: No patients have duplicate rows with identical anatomical measurements.


In [ ]:
import pandas as pd

# 1. Prepare unique keys for Master and Metadata
# We use PATNO and a YYYYMMDD formatted date for matching
df_master['PATNO_str'] = df_master['PATNO'].astype(str)
df_master['Match_Date'] = pd.to_datetime(df_master['Study_Date'], format='%m-%d-%y').dt.strftime('%Y%m%d')
master_keys = set(zip(df_master['PATNO_str'], df_master['Match_Date']))

df_metadata['PATNO_str'] = df_metadata['Subject ID'].astype(str)
df_metadata['Match_Date'] = pd.to_datetime(df_metadata['Study Date'], format='%m-%d-%y').dt.strftime('%Y%m%d')
meta_keys = set(zip(df_metadata['PATNO_str'], df_metadata['Match_Date']))

# 2. Find discrepancies
only_in_master = master_keys - meta_keys
only_in_meta = meta_keys - master_keys

print(f"--- Discrepancy Audit ---")
print(f"Total unique keys in Master: {len(master_keys)}")
print(f"Total unique keys in Metadata: {len(meta_keys)}")

if only_in_master:
    print(f"\nSessions in Master NOT in Metadata ({len(only_in_master)}):")
    for k in only_in_master: print(f"  - PATNO: {k[0]}, Date: {k[1]}")

if only_in_meta:
    print(f"\nSessions in Metadata NOT in Master ({len(only_in_meta)}):")
    for k in only_in_meta: print(f"  - PATNO: {k[0]}, Date: {k[1]}")

# 3. Check for the 3866 manual session explicitly
sub_3866 = df_master[df_master['PATNO_str'] == '3866']
print(f"\nSessions found in Master for Subject 3866 ({len(sub_3866)}):")
print(sub_3866[['PATNO', 'EVENT_ID', 'Study_Date', 'Image_Description']])

--- Discrepancy Audit ---
Total unique keys in Master: 1033
Total unique keys in Metadata: 1036

Sessions in Metadata NOT in Master (3):
  - PATNO: 3323, Date: 20160810
  - PATNO: 3309, Date: 20151215
  - PATNO: 3866, Date: 20140926

Sessions found in Master for Subject 3866 (2):
     PATNO EVENT_ID Study_Date Image_Description
1027  3866       BL   07-20-12     MPRAGE GRAPPA
1028  3866      V04   09-04-13     MPRAGE GRAPPA


In [ ]:
import pandas as pd

# 1. Determine ratios
subjects_to_fix = ['3323', '3309']
for sid in subjects_to_fix:
    expected = len(df_metadata[df_metadata['Subject ID'].astype(str) == sid])
    # Check how many rows exist in master for this PATNO
    actual = len(df_master[df_master['PATNO'].astype(str) == sid])

    status_label = f"{actual}/{expected} completed"

    # 2. Update Checkpoint
    idx = df_checkpoint[df_checkpoint['Subject ID'].astype(str) == sid].index
    if not idx.empty:
        df_checkpoint.at[idx[0], 'status'] = status_label
        print(f"Updated Subject {sid}: Status set to '{status_label}'")
    else:
        print(f"Warning: Subject {sid} not found in checkpoint.")

# 3. Save to Drive
df_checkpoint.to_csv(checkpoint_path, index=False)
print(f"\nCheckpoint saved to: {checkpoint_path}")
display(df_checkpoint[df_checkpoint['Subject ID'].astype(str).isin(subjects_to_fix)])

Updated Subject 3323: Status set to '2/3 completed'
Updated Subject 3309: Status set to '3/4 completed'

Checkpoint saved to: /content/drive/MyDrive/Fastsurfer_Automated_Pipeline/processing_checkpoint_multilcmm_longitudinal.csv


,Subject ID,status,timestamp,nipoppy BL
301,3309,3/4 completed,2026-03-31 14:10:45,Yes
304,3323,2/3 completed,2026-03-31 13:27:55,Yes


In [ ]:
master_path = '/content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv'

df_master = pd.read_csv(master_path)

In [ ]:
import pandas as pd

# Check for duplicate Patient/Date combinations in the master sheet
# This identifies if one visit has multiple processed outputs
df_master['PATNO_str'] = df_master['PATNO'].astype(str)
key_duplicates = df_master[df_master.duplicated(subset=['PATNO_str', 'Study_Date'], keep=False)]

print(f'--- Audit: Duplicate Visit Entries in Master ({len(key_duplicates)} rows) ---')
if not key_duplicates.empty:
    # Display the identifying info and a few structure volumes to see if they differ
    display(key_duplicates[['PATNO', 'EVENT_ID', 'Study_Date', 'Image_Description', 'BrainSegVol', 'MaskVol']])
    print('\nThese rows have the same Patient and Date but different underlying data.')
else:
    print('No duplicate Patient/Date combinations found in the master sheet.')

--- Audit: Duplicate Visit Entries in Master (8 rows) ---


,PATNO,EVENT_ID,Study_Date,Image_Description,BrainSegVol,MaskVol
208,120544,V04,01-10-23,3D T1-weighted,1.039512e+06,1344103.0
209,120544,V04,01-10-23,3D T1-weighted,1.039508e+06,1344076.0
309,141135,BL,03-17-22,3D T1-weighted,1.210359e+06,1557768.0
310,141135,BL,03-17-22,3D T1-weighted,1.217840e+06,1571144.0
872,3760,BL,12-07-11,MPRAGE GRAPPA,1.183947e+06,1543478.0
873,3760,BL,12-07-11,MPRAGE GRAPPA,1.174519e+06,1529079.0
899,3780,BL,06-20-12,MPRAGE GRAPPA,1.003034e+06,1300228.0
900,3780,BL,06-20-12,MPRAGE GRAPPA,1.022268e+06,1324588.0



These rows have the same Patient and Date but different underlying data.


In [ ]:
import os
import shutil
import pandas as pd
import nibabel as nib

# 1. Reset Diagnostic Directory
diag_dir = "/content/master_duplicates_diagnostic"
if os.path.exists(diag_dir):
    shutil.rmtree(diag_dir)
os.makedirs(diag_dir, exist_ok=True)

if 'Study_Date' in key_duplicates.columns:
    key_duplicates = key_duplicates.rename(columns={'Study_Date': 'Study Date'})

target_master_duplicates = ['120544', '141135', '3760', '3780']

try:
    ppmi_diag = ppmi_downloader.PPMIDownloader(config_file=CONFIG_FILE, headless=True)

    for sid in target_master_duplicates:
        print(f"\n{'='*70}\nAUDIT FOR SUBJECT: {sid}\n{'='*70}")

        dest = os.path.join(diag_dir, sid)
        os.makedirs(dest, exist_ok=True)

        duplicated_dates = key_duplicates[key_duplicates['PATNO_str'] == sid]['Study Date'].unique()
        cohort = df_metadata[(df_metadata['Subject ID'].astype(str) == sid) &
                             (df_metadata['Study Date'].isin(duplicated_dates))].copy()

        if cohort.empty:
            print(f"No metadata found for {sid}")
            continue

        cohort['PATNO'] = cohort['Subject ID']
        cohort['EVENT_ID'] = cohort.get('Visit_Short', cohort.get('Visit'))

        print(f"[Step 1] Downloading DICOMs for {sid}...")
        ppmi_diag.download_imaging_data(cohort[['PATNO', 'EVENT_ID', 'Description', 'Image ID']], destination_dir=dest)

        # 2. PRE-CONVERSION CLEANUP
        ppmi_root = os.path.join(dest, 'PPMI')
        if os.path.exists(ppmi_root):
            print(f"[Step 2] Cleaning PPMI directory for {sid}...")
            for item in os.listdir(ppmi_root):
                item_path = os.path.join(ppmi_root, item)
                # Keep it if the subject ID is anywhere in the filename/foldername
                if sid not in str(item):
                    print(f"  Removing irrelevant item: {item}")
                    if os.path.isdir(item_path):
                        shutil.rmtree(item_path)
                    else:
                        os.remove(item_path)
        else:
            print("[Warning] No PPMI folder found in dest.")

        # 3. CONVERSION
        print(f"[Step 3] Converting unique DICOMs to NIfTI...")
        # Suppress output for readability but run the conversion
        os.system(f"dcm2niix -z y -f %p_S%s_%t -o {dest} {dest} > /dev/null")

        # 4. AUDIT
        print(f"Unique files found for {sid}:")
        n_files = sorted([f for f in os.listdir(dest) if f.endswith('.nii.gz')])
        for n in n_files:
            fpath = os.path.join(dest, n)
            img = nib.load(fpath)
            hdr = img.header
            size_mb = os.path.getsize(fpath) / (1024*1024)
            print(f" - {n} ({size_mb:.2f} MB) | Dims: {img.shape}")

    ppmi_diag.driver.quit()
    print("\n--- Clean Diagnostic Run Complete ---")
except Exception as e:
    print(f"Diagnostic failed: {e}")

Using Chrome binary at: /usr/bin/chromium-browser


LivingPark-utils|DEBUG|ppmi_downloader.py:173 in __init__()
                       self.driver.capabilities: {'acceptInsecureCerts': False,
                                                  'browserName': 'chrome',
                                                  'browserVersion': '108.0.5359.40',
                                                  'chrome': {'chromedriverVersion': '108.0.5359.40 '
                                                                                    '(280b5fcaab3e877562b06cfaf2eb51121e13c3b9-refs/branch-heads/5359@{#689})',
                                                             'userDataDir': '/tmp/.org.chromium.Chromium.g8ildO'},
                                                  'goog:chromeOptions': {'debuggerAddress': 'localhost:41451'},
                                                  'networkConnectionEnabled': False,
                                                  'pageLoadStrategy': 'normal',
                                               


AUDIT FOR SUBJECT: 120544
[Step 1] Downloading DICOMs for 120544...
Logging in and navigating to search...


LivingPark-utils|DEBUG|ppmi_navigator.py:221 in click_button()
                       "Click button": 'Click button'
                       field: 'ida-cookie-policy-accept'
                       debug_name: 'Cookie Policy'
                       caller: 'ppmi_navigator.py:332 validate_cookie_policy'
                       current_url: 'https://ida.loni.usc.edu/login.jsp?project=PPMI'
LivingPark-utils|DEBUG|ppmi_navigator.py:124 in wait_for_element_to_be_visible()
                       "Wait for element to be visible": 'Wait for element to be visible'
                       field: 'ida-menu-option.sub-menu.user'
                       BY: 'class name'
                       debug_name: ''
LivingPark-utils|DEBUG|ppmi_navigator.py:124 in wait_for_element_to_be_visible()
                       "Wait for element to be visible": 'Wait for element to be visible'
                       field: 'userEmail'
                       BY: 'name'
                       debug_name: ''
LivingPark-util

Navigating to Export page for 120544...

--- Attempt 1 ---
Selecting target scans via scrolling...
  Checked: ('120544', 'V04', '3D T1-weighted')
All target scans successfully selected.
Initiating download overlay...
Found 2 zip link(s) with labels: ['Zip File', '']
  Triggered download for: Zip File...
  Skipping link 2 (empty label)...
  Metadata download triggered.
Waiting for files to finish downloading...


...

Download and extraction complete.
[Step 2] Cleaning PPMI directory for 3760...
  Removing irrelevant item: PPMI_141135_3D_T1-weighted_S1129123_I1575242.xml
  Removing irrelevant item: 141135
  Removing irrelevant item: 120544
  Removing irrelevant item: PPMI_120544_3D_T1-weighted_S1204711_I1678078.xml
[Step 3] Converting unique DICOMs to NIfTI...
Unique files found for 3760:
 - MPRAGE_GRAPPA_S8_20111207093557.nii.gz (11.12 MB) | Dims: (240, 256, 175)
 - MPRAGE_GRAPPA_S8_20111207093557_Eq_1.nii.gz (11.15 MB) | Dims: (240, 256, 175)

AUDIT FOR SUBJECT: 3780
[Step 1] Downloading DICOMs for 3780...
Logging in and navigating to search...


LivingPark-utils|DEBUG|ppmi_navigator.py:328 in validate_cookie_policy()
                       "Cookie Policy already accepted": 'Cookie Policy already accepted'
LivingPark-utils|DEBUG|ppmi_navigator.py:124 in wait_for_element_to_be_visible()
                       "Wait for element to be visible": 'Wait for element to be visible'
                       field: 'ida-menu-option.sub-menu.user'
                       BY: 'class name'
                       debug_name: ''
LivingPark-utils|DEBUG|ppmi_navigator.py:379 in login()
                       "Already logged in": 'Already logged in'
LivingPark-utils|DEBUG|ppmi_navigator.py:814 in predicate()
                       select_all: <selenium.webdriver.remote.webelement.WebElement (session="d8b0c825f961673dff1d367e5aca44e7", element="7b9f40eb-c81c-4ed9-902d-02432704a7b7")>
                       select_all.is_selected(): False
LivingPark-utils|DEBUG|ppmi_navigator.py:815 in predicate()
                       add_to_collection: <selenium.w

Navigating to Export page for 3780...

--- Attempt 1 ---
Selecting target scans via scrolling...
  Checked: ('3780', 'BL', 'MPRAGE GRAPPA')
All target scans successfully selected.
Initiating download overlay...
Found 2 zip link(s) with labels: ['Zip File', '']
  Triggered download for: Zip File...
  Skipping link 2 (empty label)...
  Metadata download triggered.
Waiting for files to finish downloading...


  0%|          | 0/9 [00:00<?, ?it/s]LivingPark-utils|INFO|f"Successfully downloaded files {filesname}...": 'Successfully downloaded files ['PPMI/3760/MPRAGE_GRAPPA/2011-12-07_09_35_57.0/I287999/S142620I287999.xml', 'PPMI/PPMI_3760_MPRAGE_GRAPPA_S142620_I287999.xml']...'
LivingPark-utils|INFO|f"Successfully downloaded files {filesname}...": 'Successfully downloaded files ['PPMI/141135/3D_T1-weighted/2022-03-17_13_42_36.0/I1575242/PPMI_141135_MR_3D_T1-weighted_br_raw_20220429030142908_179_S1129123_I1575242.dcm', 'PPMI/141135/3D_T1-weighted/2022-03-17_13_42_36.0/I1575242/PPMI_141135_MR_3D_T1-weighted_br_raw_20220429030154245_138_S1129123_I1575242.dcm']...'
 22%|██▏       | 2/9 [00:00<00:01,  4.05it/s]LivingPark-utils|INFO|f"Successfully downloaded files {filesname}...": 'Successfully downloaded files ['PPMI/3780/MPRAGE_GRAPPA/2012-06-20_10_09_37.0/I365070/S185630I365070.xml', 'PPMI/PPMI_3780_MPRAGE_GRAPPA_S185630_I365070.xml']...'
LivingPark-utils|INFO|f"Successfully downloaded files {fi

Download and extraction complete.
[Step 2] Cleaning PPMI directory for 3780...
  Removing irrelevant item: PPMI_141135_3D_T1-weighted_S1129123_I1575242.xml
  Removing irrelevant item: 141135
  Removing irrelevant item: 120544
  Removing irrelevant item: 3760
  Removing irrelevant item: PPMI_3760_MPRAGE_GRAPPA_S142620_I287999.xml
  Removing irrelevant item: PPMI_120544_3D_T1-weighted_S1204711_I1678078.xml
[Step 3] Converting unique DICOMs to NIfTI...
Unique files found for 3780:
 - MPRAGE_GRAPPA_S9_20120620100937.nii.gz (10.75 MB) | Dims: (240, 256, 174)
 - MPRAGE_GRAPPA_S9_20120620100937_Eq_1.nii.gz (10.83 MB) | Dims: (240, 256, 175)

--- Clean Diagnostic Run Complete ---


In [ ]:
import os
import nibabel as nib
import pandas as pd

diag_dir = '/content/master_duplicates_diagnostic'
subjects = ['120544', '141135', '3760', '3780']

summary_data = []

print('--- Diagnostic Summary: NIfTI File Properties ---')

for sid in subjects:
    subject_path = os.path.join(diag_dir, sid)
    if os.path.exists(subject_path):
        niftis = sorted([f for f in os.listdir(subject_path) if f.endswith('.nii.gz')])

        if not niftis:
            print(f'\nSubject {sid}: No NIfTI files found.')
            continue

        for n in niftis:
            fpath = os.path.join(subject_path, n)
            img = nib.load(fpath)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)

            summary_data.append({
                'Subject ID': sid,
                'Filename': n,
                'Dimensions': str(img.shape),
                'Voxel Size': str(img.header.get_zooms()),
                'Size (MB)': f'{size_mb:.2f}'
            })

if summary_data:
    df_summary = pd.DataFrame(summary_data)
    display(df_summary)
else:
    print('No data available to summarize. Please ensure the diagnostic loop has run successfully.')

--- Diagnostic Summary: NIfTI File Properties ---


,Subject ID,Filename,Dimensions,Voxel Size,Size (MB)
0,120544,Anon_S201_20230110092443.nii.gz,"(192, 256, 256)","(np.float32(1.0), np.float32(1.0), np.float32(...",8.41
1,120544,Anon_S201_20230110092443a.nii.gz,"(192, 256, 256)","(np.float32(1.0), np.float32(1.0), np.float32(...",8.41
2,141135,Anon_S2_20220317134237.nii.gz,"(256, 256, 193)","(np.float32(1.0), np.float32(1.0), np.float32(...",12.38
3,141135,Anon_S2_20220317134237_Eq_1.nii.gz,"(256, 256, 191)","(np.float32(1.0), np.float32(1.0), np.float32(...",12.27
4,3760,MPRAGE_GRAPPA_S8_20111207093557.nii.gz,"(240, 256, 175)","(np.float32(1.0), np.float32(1.0), np.float32(...",11.12
5,3760,MPRAGE_GRAPPA_S8_20111207093557_Eq_1.nii.gz,"(240, 256, 175)","(np.float32(1.0), np.float32(1.0), np.float32(...",11.15
6,3780,MPRAGE_GRAPPA_S9_20120620100937.nii.gz,"(240, 256, 174)","(np.float32(1.0), np.float32(1.0), np.float32(...",10.75
7,3780,MPRAGE_GRAPPA_S9_20120620100937_Eq_1.nii.gz,"(240, 256, 175)","(np.float32(1.0), np.float32(1.0), np.float32(...",10.83


In [ ]:
import numpy as np
import nibabel as nib
import os

# Target subject with apparent duplicates
sid = '120544'
subject_path = '/content/master_duplicates_diagnostic/120544'
f1_path = os.path.join(subject_path, 'Anon_S201_20230110092443.nii.gz')
f2_path = os.path.join(subject_path, 'Anon_S201_20230110092443a.nii.gz')

if os.path.exists(f1_path) and os.path.exists(f2_path):
    img1 = nib.load(f1_path)
    img2 = nib.load(f2_path)

    data1 = img1.get_fdata()
    data2 = img2.get_fdata()

    print(f'--- Voxel-Level Comparison for Subject {sid} ---')

    # Check if data arrays are exactly the same
    if np.array_equal(data1, data2):
        print('RESULT: The voxel data is IDENTICAL (MSE = 0).')
    else:
        mse = np.mean((data1 - data2) ** 2)
        diff_count = np.sum(data1 != data2)
        print(f'RESULT: The data is DIFFERENT.')
        print(f' - Mean Squared Error: {mse}')
        print(f' - Number of differing voxels: {diff_count} out of {data1.size}')

    # Check headers for metadata differences
    print('\n--- Header Metadata Check ---')
    h1, h2 = img1.header, img2.header

    # Check key fields: sform/qform codes and descriptions
    print(f"File 1 descrip: {h1['descrip'].tobytes().decode().strip()}")
    print(f"File 2 descrip: {h2['descrip'].tobytes().decode().strip()}")

    if not np.allclose(img1.affine, img2.affine):
        print('WARNING: Affine matrices differ (indicates different spatial orientation).')
    else:
        print('Affine matrices are identical.')
else:
    print('Files not found for comparison. Please run the diagnostic loop first.')

--- Voxel-Level Comparison for Subject 120544 ---
RESULT: The voxel data is IDENTICAL (MSE = 0).

--- Header Metadata Check ---
File 1 descrip: TE=2.9;Time=0.000                                                               
File 2 descrip: TE=2.9;Time=93058.410                                                           


In [ ]:
import scipy.stats as stats
import numpy as np

# Re-identify structure columns (index 13 onwards in the template structure)
# We exclude identity columns and global measures
identity_cols = ['PATNO', 'EVENT_ID', 'Study_Date', 'Image_Description']
global_measures = ['MaskVol', 'BrainSegVol', 'BrainSegVolNotVent', 'SupraTentorialVol', 'SupraTentorialVolNotVent', 'SubCortGrayVol', 'rhCerebralWhiteMatterVol', 'lhCerebralWhiteMatterVol', 'CerebralWhiteMatterVol']

# Dynamically identify anatomical structure columns present in df_master
structures = [c for c in df_master.columns if c not in identity_cols + global_measures + ['PATNO_str', 'Match_Date']]

audit_subjects = ['120544', '141135', '3760', '3780']
correlation_results = []

print(f'--- Refined Correlation Audit: Duplicated Session Pairs ({len(structures)} structures) ---')

for sid in audit_subjects:
    sub_df = df_master[df_master['PATNO'].astype(str) == sid]
    # Ensure consistent date column name
    date_col = 'Study_Date' if 'Study_Date' in sub_df.columns else 'Study Date'
    date_counts = sub_df[date_col].value_counts()
    dup_dates = date_counts[date_counts > 1].index

    for d in dup_dates:
        pair = sub_df[sub_df[date_col] == d]

        if len(pair) == 2:
            vols_1 = pair.iloc[0][structures].values.astype(float)
            vols_2 = pair.iloc[1][structures].values.astype(float)

            corr, _ = stats.pearsonr(vols_1, vols_2)
            mape = np.mean(np.abs((vols_1 - vols_2) / (vols_1 + 1e-9))) * 100

            correlation_results.append({
                'Subject ID': sid,
                'Date': d,
                'Pearson R': round(corr, 6),
                'MAPE (%)': round(mape, 4),
                'Image 1': pair.iloc[0]['Image_Description'],
                'Image 2': pair.iloc[1]['Image_Description']
            })
        else:
            print(f'Subject {sid} on {d} has {len(pair)} rows, skipping automated pair audit.')

if correlation_results:
    df_corr = pd.DataFrame(correlation_results)
    display(df_corr)
    print(f'\nMean Correlation: {df_corr["Pearson R"].mean():.6f}')
else:
    print("No duplicated session pairs found to correlate.")

--- Refined Correlation Audit: Duplicated Session Pairs (100 structures) ---


,Subject ID,Date,Pearson R,MAPE (%),Image 1,Image 2
0,120544,01-10-23,1.00000,0.0536,3D T1-weighted,3D T1-weighted
1,141135,03-17-22,0.99999,1.5331,3D T1-weighted,3D T1-weighted
2,3760,12-07-11,0.99998,1.5710,MPRAGE GRAPPA,MPRAGE GRAPPA
3,3780,06-20-12,0.99997,2.2593,MPRAGE GRAPPA,MPRAGE GRAPPA



Mean Correlation: 0.999985


In [ ]:
import pandas as pd
import numpy as np

# Filter for the specific subject and date
dup_120544 = df_master[(df_master['PATNO'].astype(str) == '120544') & (df_master['Study_Date'] == '01-10-23')]

print('--- Detailed Comparison for Subject 120544 Duplicates ---')
if len(dup_120544) == 2:
    row1 = dup_120544.iloc[0]
    row2 = dup_120544.iloc[1]

    # Compare all numeric columns
    diffs = {}
    for col in df_master.columns:
        val1 = row1[col]
        val2 = row2[col]
        if isinstance(val1, (int, float, np.number)):
            if not np.isclose(val1, val2, atol=1e-9):
                diffs[col] = (val1, val2, val1 - val2)

    if diffs:
        print(f'Found {len(diffs)} columns with numerical differences:')
        df_diffs = pd.DataFrame.from_dict(diffs, orient='index', columns=['Value_Row1', 'Value_Row2', 'Delta'])
        display(df_diffs)
    else:
        print('No numerical differences found at 1e-9 precision.')

    # Check non-numeric columns
    print('\n--- Non-Numeric Metadata Check ---')
    for col in ['EVENT_ID', 'Image_Description']:
        print(f'{col}: Row1="{row1[col]}", Row2="{row2[col]}"')
else:
    print(f'Expected 2 rows for 120544 on 01-10-23, found {len(dup_120544)}.')

--- Detailed Comparison for Subject 120544 Duplicates ---
Found 99 columns with numerical differences:


,Value_Row1,Value_Row2,Delta
MaskVol,1.344103e+06,1.344076e+06,27.000000
SubCortGrayVol,5.167506e+04,5.168544e+04,-10.373863
rhCerebralWhiteMatterVol,2.074423e+05,2.074512e+05,-8.857453
lhCerebralWhiteMatterVol,2.093044e+05,2.093003e+05,4.178753
CerebralWhiteMatterVol,4.167468e+05,4.167515e+05,-4.678700
...,...,...,...
ctx-rh-superiorparietal,8.793792e+03,8.788540e+03,5.252000
ctx-rh-superiortemporal,1.390411e+04,1.391193e+04,-7.815000
ctx-rh-supramarginal,9.094103e+03,9.091579e+03,2.524000
ctx-rh-transversetemporal,7.783880e+02,7.791110e+02,-0.723000



--- Non-Numeric Metadata Check ---
EVENT_ID: Row1="V04", Row2="V04"
Image_Description: Row1="3D T1-weighted", Row2="3D T1-weighted"


In [ ]:
import pandas as pd
import os

# 1. Filter out the intensity-equalized ('Eq_1') rows based on Image_Description
# Note: From our diagnostic, 'Eq_1' versions are usually the duplicates we want to drop.
df_cleaned = df_master[~df_master['Image_Description'].str.contains('_Eq_1', na=False)].copy()

# 2. Specifically deduplicate Subject 120544 on the date 01-10-23
# We keep the first occurrence as the data is voxel-identical.
df_cleaned = df_cleaned.drop_duplicates(subset=['PATNO', 'Study_Date'], keep='first')

# 3. Add Subject 3866's 3rd scan (2014-09-26)
# We previously processed this manually as '3866_MPRAGE_GRAPPA_2014'
# Let's extract that row from the aggregation logic or the folder if it wasn't in df_master

# Check if 3866's 2014 scan is missing from current cleaned df
if not ((df_cleaned['PATNO'].astype(str) == '3866') & (df_cleaned['Study_Date'] == '09-26-14')).any():
    print("Adding missing 2014 scan for Subject 3866...")
    # We construct the row metadata from the known manual fix info
    # Since the stats are in the zip, we'll assume it was aggregated but maybe filtered/missing
    # If it's not in df_master at all, we'll re-run a mini aggregation for just that folder

    # For the sake of this step, we'll ensure 3866 is updated.
    # (In a real scenario, we'd pull the specific row from all_rows before df_master creation)

# 4. Verification
final_count = len(df_cleaned)
print(f'--- Final Aggregation Audit ---')
print(f'Final Row Count: {final_count}')

if final_count == 1035:
    print('SUCCESS: Target of 1035 entries reached.')
else:
    print(f'NOTICE: Current count is {final_count}. Check for remaining duplicates or missing entries.')

# Save the final version
final_master_path = '/content/drive/MyDrive/data/FastSurfer_Master_Aggregation_Final.csv'
df_cleaned.to_csv(final_master_path, index=False)
display(df_cleaned.head())

Adding missing 2014 scan for Subject 3866...
--- Final Aggregation Audit ---
Final Row Count: 1033
NOTICE: Current count is 1033. Check for remaining duplicates or missing entries.


,PATNO,EVENT_ID,Study_Date,Image_Description,MaskVol,BrainSegVol,BrainSegVolNotVent,SupraTentorialVol,SupraTentorialVolNotVent,SubCortGrayVol,...,ctx-rh-precuneus,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-transversetemporal,ctx-rh-insula,PATNO_str
0,100001,BL,10-07-20,SAG 3D MPRAGE,1426491.0,1.106751e+06,1.083366e+06,9.676464e+05,9.442612e+05,57988.070480,...,8055.184,2290.155,9682.411,25368.864,7964.390,13764.618,7987.321,862.207,5872.390,100001
1,100001,V06,11-29-22,SAG 3D MPRAGE,1408328.0,1.089433e+06,1.065102e+06,9.520664e+05,9.277358e+05,56786.189847,...,8092.248,2238.521,9628.452,24891.809,7938.342,13677.405,7947.545,839.395,5757.502,100001
2,100001,V10,09-11-24,SAG 3D T1 FSPGR,1446766.0,1.134900e+06,1.107924e+06,9.917805e+05,9.648043e+05,58324.505092,...,7794.166,2495.664,10490.085,26054.235,7785.494,14174.507,8643.477,829.227,5873.501,100001
3,100005,BL,01-27-21,3D T1 _weighted,1707678.0,1.357007e+06,1.335605e+06,1.189561e+06,1.168159e+06,71864.305987,...,11949.686,2701.991,11397.665,27442.435,12118.098,17765.782,11386.644,978.776,6785.774,100005
4,100005,V04,02-02-22,3D T1 _weighted,1703032.0,1.345458e+06,1.323268e+06,1.180172e+06,1.157982e+06,71555.081236,...,11951.343,2637.227,11004.912,27114.055,11946.268,17658.720,10862.961,1003.884,6634.855,100005


In [ ]:
import pandas as pd
import os
import zipfile
import re

# Configuration needed for this cell
drive_results_dir = '/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/'

# 1. Start with the cleaned data (excluding Eq_1 and duplicates)
df_final = df_master[~df_master['Image_Description'].str.contains('_Eq_1', na=False)].copy()
df_final = df_final.drop_duplicates(subset=['PATNO', 'Study_Date'], keep='first')

# 2. Manual Extraction for Subject 3866 Session 3 (2014)
subject_id = '3866'
target_folder = '3866_MPRAGE_GRAPPA_2014'
zip_path = os.path.join(drive_results_dir, f'stats_{subject_id}.zip')

print(f"--- Manually adding missing 2014 session for {subject_id} ---")
try:
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Find the specific stats file
        matches = [f for f in z.namelist() if f'{target_folder}/aseg+DKT.stats' in f]
        if matches:
            stats_path = matches[0]
            with z.open(stats_path) as f:
                content = f.read().decode('utf-8')
                lines = content.split('\n')

                # Create the data row
                new_row = {
                    'PATNO': int(subject_id),
                    'EVENT_ID': 'V06',
                    'Study_Date': '09-26-14',
                    'Image_Description': 'MPRAGE GRAPPA'
                }

                # Parse Global Measures (using global_measures list from environment)
                for gm in global_measures:
                    m_pattern = re.compile(rf'Measure\s+.*?\s+{gm},\s+.*?\s+([\d\.]+),\s+mm\^3', re.IGNORECASE)
                    m_match = m_pattern.search(content)
                    new_row[gm] = float(m_match.group(1)) if m_match else None

                # Parse Structures (using structures list from environment)
                start_table = False
                for line in lines:
                    if line.startswith('# ColHeaders'):
                        start_table = True
                        continue
                    if start_table and line.strip() and not line.startswith('#'):
                        parts = line.split()
                        if len(parts) >= 5 and parts[4] in structures:
                            new_row[parts[4]] = float(parts[3])

                # Append to dataframe
                df_final = pd.concat([df_final, pd.DataFrame([new_row])], ignore_index=True)
                print(f"SUCCESS: Added {subject_id} 2014-09-26 session.")
        else:
            print(f"Error: Folder {target_folder} not found in {zip_path}")

except Exception as e:
    print(f"Manual addition failed: {e}")

# 3. Final Sort and Count
df_final['Study_Date_Dt'] = pd.to_datetime(df_final['Study_Date'], format='%m-%d-%y')
df_final = df_final.sort_values(by=['PATNO', 'Study_Date_Dt']).drop(columns=['Study_Date_Dt'])

print(f"\nFinal Row Count (excluding header): {len(df_final)}")
if len(df_final) == 1034:
    print("TARGET REACHED: 1034 sessions accounted for.")

# Save Final File
final_csv_path = '/content/drive/MyDrive/data/FastSurfer_Master_Aggregation_FINAL_CLEAN.csv'
os.makedirs(os.path.dirname(final_csv_path), exist_ok=True)
df_final.to_csv(final_csv_path, index=False)
print(f"Saved to: {final_csv_path}")
display(df_final[df_final['PATNO'] == 3866])

--- Manually adding missing 2014 session for 3866 ---
SUCCESS: Added 3866 2014-09-26 session.

Final Row Count (excluding header): 1034
TARGET REACHED: 1034 sessions accounted for.
Saved to: /content/drive/MyDrive/data/FastSurfer_Master_Aggregation_FINAL_CLEAN.csv


,PATNO,EVENT_ID,Study_Date,Image_Description,MaskVol,BrainSegVol,BrainSegVolNotVent,SupraTentorialVol,SupraTentorialVolNotVent,SubCortGrayVol,...,ctx-rh-precuneus,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-transversetemporal,ctx-rh-insula,PATNO_str
981,3866,BL,07-20-12,MPRAGE GRAPPA,1191984.0,915923.933909,896361.609130,795520.501844,775958.177065,47850.745898,...,8556.857,2285.120,8267.457,18899.114,7991.803,11495.622,6604.726,550.996,5641.252,3866
982,3866,V04,09-04-13,MPRAGE GRAPPA,1191246.0,917005.717759,897311.878627,796032.104973,776338.265841,46938.815059,...,8553.838,2222.511,8237.576,19109.270,8201.548,11723.066,6616.370,549.329,5691.451,3866
1033,3866,V06,09-26-14,MPRAGE GRAPPA,1188328.0,908440.086192,886920.568749,788819.475578,767299.958135,45396.710148,...,8252.990,2184.000,8256.620,18650.587,8034.098,11421.196,6420.313,534.358,5593.144,NaN


In [ ]:
import pandas as pd

# 1. Drop the temporary helper column if it exists
if 'PATNO_str' in df_final.columns:
    df_final = df_final.drop(columns=['PATNO_str'])

# 2. Define the requested clean path
master_aggregation_path = '/content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv'

# 3. Save to Drive
df_final.to_csv(master_aggregation_path, index=False)

print(f'Successfully saved {len(df_final)} rows to: {master_aggregation_path}')
display(df_final.head())

Successfully saved 1034 rows to: /content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv


,PATNO,EVENT_ID,Study_Date,Image_Description,MaskVol,BrainSegVol,BrainSegVolNotVent,SupraTentorialVol,SupraTentorialVolNotVent,SubCortGrayVol,...,ctx-rh-precentral,ctx-rh-precuneus,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-transversetemporal,ctx-rh-insula
654,3102,BL,11-23-10,MPRAGE GRAPPA,1499081.0,1.182849e+06,1.145305e+06,1.057267e+06,1.019722e+06,61813.982035,...,11957.589,9126.744,2521.642,10475.416,24139.223,8973.840,14288.321,9825.417,754.275,6592.734
655,3102,V06,02-06-13,MPRAGE GRAPPA,1483386.0,1.157672e+06,1.116668e+06,1.032450e+06,9.914464e+05,61313.443194,...,11404.606,8860.182,2215.281,9615.788,23012.348,8669.595,14334.517,9461.941,726.005,6481.004
656,3107,BL,04-13-11,MPRAGE GRAPPA,1582827.0,1.221950e+06,1.175303e+06,1.080521e+06,1.033874e+06,58563.430515,...,12540.186,11784.287,2938.990,10636.854,25533.905,11035.693,15055.836,9785.279,1031.848,6778.907
657,3107,V04,03-28-12,MPRAGE GRAPPA,1574440.0,1.205167e+06,1.157359e+06,1.062168e+06,1.014360e+06,58477.053261,...,12488.316,11805.044,2390.236,9127.156,23888.689,11130.397,15124.977,9754.373,1071.908,6804.215
658,3107,V06,05-15-13,MPRAGE GRAPPA,1566161.0,1.198107e+06,1.151591e+06,1.056192e+06,1.009676e+06,58355.876078,...,12495.818,11827.559,2372.993,9222.659,23981.070,10950.503,15092.813,9621.370,1034.904,6841.712


In [ ]:
import pandas as pd

# Path to the master file on Google Drive
master_path = '/content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv'

# Load the data
df_master = pd.read_csv(master_path)

# Ensure PATNO is a string for alphabetical sorting (e.g., '100' comes before '2')
df_master['PATNO'] = df_master['PATNO'].astype(str)

# Create a temporary datetime column for chronological sorting within each subject
df_master['Study_Date_Dt'] = pd.to_datetime(df_master['Study_Date'], format='%m-%d-%y')

# Re-sort: Alphabetical by PATNO, then Chronological by Date
df_master = df_master.sort_values(by=['PATNO', 'Study_Date_Dt'])

# Drop the helper column
df_master = df_master.drop(columns=['Study_Date_Dt'])

# Save the sorted version back to Google Drive
df_master.to_csv(master_path, index=False)

print(f'Successfully re-sorted {len(df_master)} rows and saved to Drive: {master_path}')
display(df_master.head(10))

Successfully re-sorted 1034 rows and saved to Drive: /content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv


,PATNO,EVENT_ID,Study_Date,Image_Description,MaskVol,BrainSegVol,BrainSegVolNotVent,SupraTentorialVol,SupraTentorialVolNotVent,SubCortGrayVol,...,ctx-rh-precentral,ctx-rh-precuneus,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-transversetemporal,ctx-rh-insula
380,100001,BL,10-07-20,SAG 3D MPRAGE,1426491.0,1.106751e+06,1.083366e+06,9.676464e+05,9.442612e+05,57988.070480,...,10456.012,8055.184,2290.155,9682.411,25368.864,7964.390,13764.618,7987.321,862.207,5872.390
381,100001,V06,11-29-22,SAG 3D MPRAGE,1408328.0,1.089433e+06,1.065102e+06,9.520664e+05,9.277358e+05,56786.189847,...,10350.497,8092.248,2238.521,9628.452,24891.809,7938.342,13677.405,7947.545,839.395,5757.502
382,100001,V10,09-11-24,SAG 3D T1 FSPGR,1446766.0,1.134900e+06,1.107924e+06,9.917805e+05,9.648043e+05,58324.505092,...,10653.921,7794.166,2495.664,10490.085,26054.235,7785.494,14174.507,8643.477,829.227,5873.501
383,100005,BL,01-27-21,3D T1 _weighted,1707678.0,1.357007e+06,1.335605e+06,1.189561e+06,1.168159e+06,71864.305987,...,13115.193,11949.686,2701.991,11397.665,27442.435,12118.098,17765.782,11386.644,978.776,6785.774
384,100005,V04,02-02-22,3D T1 _weighted,1703032.0,1.345458e+06,1.323268e+06,1.180172e+06,1.157982e+06,71555.081236,...,12558.124,11951.343,2637.227,11004.912,27114.055,11946.268,17658.720,10862.961,1003.884,6634.855
385,100012,BL,03-02-21,3D T1-weighted,1251802.0,9.904582e+05,9.659421e+05,8.763793e+05,8.518633e+05,49790.905924,...,10831.809,9206.014,1778.884,9259.962,25798.875,8648.853,13170.113,8711.767,779.840,5621.733
386,100012,V04,02-07-22,3D T1-weighted,1243705.0,9.784840e+05,9.540132e+05,8.655738e+05,8.411030e+05,48952.750346,...,10654.682,8946.522,1775.575,9204.806,25056.703,8076.102,12510.677,8441.316,737.469,5572.541
387,100012,V06,01-09-23,3D T1-weighted,1268714.0,9.975570e+05,9.721698e+05,8.807716e+05,8.553844e+05,49871.522178,...,10590.345,9115.046,1713.545,9244.192,25578.333,8461.324,12984.500,8725.813,770.643,5718.315
388,100017,BL,12-22-20,SAG 3D MPRAGE,1467991.0,1.132210e+06,1.090452e+06,9.995677e+05,9.578089e+05,61136.813338,...,12702.119,8450.819,2188.143,11519.019,26836.739,10328.653,13836.164,8345.319,722.942,6450.806
389,100017,V04,12-14-21,SAG 3D MPRAGE,1472525.0,1.137644e+06,1.091029e+06,1.004005e+06,9.573899e+05,61893.291275,...,12593.934,9113.839,2305.791,12116.016,27869.697,10530.195,14395.434,8795.114,855.540,6643.181


## Manual Processing of 3309 and 3323

In [ ]:
import os
import shutil
import zipfile
import glob
import re
import datetime

# --- Configuration for Manual Fix ---
manual_subjects = {
    '3309': ['3309.zip', '3309_IDA_Metadata.zip'],
    '3323': ['3323.zip', '3323_IDA_Metadata.zip']
}

for subject_id, zip_list in manual_subjects.items():
    print(f"\n{'='*80}\nMANUAL FIX: SUBJECT {subject_id}\n{'='*80}")

    working_dir = f"/content/data/{subject_id}"
    if os.path.exists(working_dir): shutil.rmtree(working_dir)
    os.makedirs(working_dir, exist_ok=True)

    # 1. Unpack
    for z_name in zip_list:
        z_path = f"/content/{z_name}"
        if os.path.exists(z_path):
            print(f"  Extracting {z_name}...")
            with zipfile.ZipFile(z_path, 'r') as z_ref: z_ref.extractall(working_dir)
        else:
            print(f"  Warning: {z_name} not found in /content/")

    # 2. Convert
    print("  Converting DICOM to NIfTI...")
    os.system(f"dcm2niix -z y -f %p_%t -o {working_dir} {working_dir} > /dev/null")

    # 3. Run FastSurfer
    niftis = sorted(glob.glob(os.path.join(working_dir, "*.nii.gz")))
    if not niftis:
        print(f"  Error: No NIfTI found for {subject_id}")
        continue

    for img in niftis:
        # Extract Year and Description from filename (e.g., MPRAGE_GRAPPA_20151215...)
        fname = os.path.basename(img)
        year_match = re.search(r'(20\d{2})', fname)
        year = year_match.group(1) if year_match else "UnknownYear"

        # Standardize SID: PATNO_Description_Year
        # We strip common dcm2niix suffixes for the description part
        clean_desc = fname.split('_20')[0].replace('.nii.gz', '')
        session_id = f"{subject_id}_{clean_desc}_{year}"

        print(f"  Processing Session: {session_id}")

        fs_cmd = (
            f"export FASTSURFER_HOME={FASTSURFER_HOME}; "
            f"{FASTSURFER_HOME}/run_fastsurfer.sh "
            f"--t1 {img} --sd {SETUP_DIR}fastsurfer_seg --sid {session_id} "
            f"--seg_only --py python3 --3T --allow_root"
        )

        try:
            # Using the v2 robust wrapper to handle potential segmentation errors
            run_fastsurfer_robust_v2(fs_cmd)

            # 4. Merge into Drive Stats ZIP
            drive_zip = f"/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/stats_{subject_id}.zip"
            staging = f"/content/merge_{subject_id}"
            if os.path.exists(staging): shutil.rmtree(staging)
            os.makedirs(staging)

            if os.path.exists(drive_zip):
                with zipfile.ZipFile(drive_zip, 'r') as z: z.extractall(staging)

            new_stats = os.path.join(SETUP_DIR, "fastsurfer_seg", session_id, "stats")
            if os.path.exists(new_stats):
                shutil.copytree(new_stats, os.path.join(staging, session_id))
                shutil.make_archive(f"/content/stats_{subject_id}", 'zip', staging)
                shutil.move(f"/content/stats_{subject_id}.zip", drive_zip)
                print(f"  SUCCESS: Stats for {session_id} merged to Drive.")

        except Exception as e:
            print(f"  Failed processing {session_id}: {e}")

print("\nManual processing sequence complete.")


MANUAL FIX: SUBJECT 3309
  Extracting 3309.zip...
  Extracting 3309_IDA_Metadata.zip...
  Converting DICOM to NIfTI...
  Processing Session: 3309_MPRAGE_GRAPPA_2015
Executing command:
export FASTSURFER_HOME=/content/fastsurfer/; /content/fastsurfer//run_fastsurfer.sh --t1 /content/data/3309/MPRAGE_GRAPPA_20151215104056.nii.gz --sd /content/fastsurfer_seg --sid 3309_MPRAGE_GRAPPA_2015 --seg_only --py python3 --3T --allow_root

--- Output Log (Robust v2.2) ---
time command failing, not using time...
  running FastSurfer as root, because it will lead to files and folders created as root.
  If you are running FastSurfer in a docker container, you can specify the user
  with '-u $(id -u):$(id -g)' (see https://docs.docker.com/engine/reference/run/#user).
Start of the log for a new run_fastsurfer.sh invocation
Version: 2.4.2+7e53343
Mon Apr  6 05:42:37 PM UTC 2026

Log file for FastSurfer pipeline, run_fastsurfer.sh and segmentation(s)
python3 /content/fastsurfer//FastSurferCNN/run_predicti

In [ ]:
import pandas as pd
import os
import zipfile
import re

# 1. Load the current state of the master sheet
master_path = '/content/drive/MyDrive/data/FastSurfer_Master_Aggregation.csv'
df_final = pd.read_csv(master_path)

# 2. Identify existing columns
all_cols = df_final.columns.tolist()
identity_cols = ['PATNO', 'EVENT_ID', 'Study_Date', 'Image_Description']
global_measures_list = [
    'MaskVol', 'BrainSegVol', 'BrainSegVolNotVent',
    'SupraTentorialVol', 'SupraTentorialVolNotVent',
    'SubCortGrayVol', 'rhCerebralWhiteMatterVol',
    'lhCerebralWhiteMatterVol', 'CerebralWhiteMatterVol'
]
structures_list = [c for c in all_cols if c not in identity_cols + global_measures_list]

# 3. Manual Extraction for 3309 and 3323 (Re-run with fixed regex)
manual_targets = {
    '3309': {'folder': '3309_MPRAGE_GRAPPA_2015', 'date': '12-15-15', 'visit': 'V08', 'desc': 'MPRAGE GRAPPA'},
    '3323': {'folder': '3323_MPRAGE_GRAPPA_2016', 'date': '08-10-16', 'visit': 'V08', 'desc': 'MPRAGE GRAPPA'}
}

drive_results_dir = '/content/drive/MyDrive/Fastsurfer_Automated_Pipeline/'

for sid, info in manual_targets.items():
    # First, remove the previously failed incomplete rows if they exist to avoid duplicates
    df_final = df_final[~((df_final['PATNO'].astype(str) == str(sid)) & (df_final['Study_Date'] == info['date']))]

    zip_path = os.path.join(drive_results_dir, f'stats_{sid}.zip')
    if not os.path.exists(zip_path): continue

    print(f"Extracting manual session for {sid} with corrected regex...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            stats_file = [f for f in z.namelist() if f"{info['folder']}/aseg+DKT.stats" in f]
            if stats_file:
                with z.open(stats_file[0]) as f:
                    content = f.read().decode('utf-8')
                    lines = content.split('\n')

                    new_row = {
                        'PATNO': str(sid),
                        'EVENT_ID': info['visit'],
                        'Study_Date': info['date'],
                        'Image_Description': info['desc']
                    }

                    # Improved Regex for Global Measures (matches: Measure BrainSeg, BrainSegVol, ..., 1134900.49, mm^3)
                    for gm in global_measures_list:
                        m_pattern = re.compile(rf'Measure\s+.*?\s+{gm},\s+.*?\s+([\d\.]+),\s+mm\^3', re.IGNORECASE)
                        m_match = m_pattern.search(content)
                        if m_match:
                            new_row[gm] = float(m_match.group(1))
                        else:
                            # Fallback pattern for different spacing/caret formatting
                            alt_pattern = re.compile(rf'{gm},\s+.*?\s+([\d\.]+)', re.IGNORECASE)
                            alt_match = alt_pattern.search(content)
                            new_row[gm] = float(alt_match.group(1)) if alt_match else None

                    # Parse Structures
                    start_table = False
                    for line in lines:
                        if line.startswith('# ColHeaders'): start_table = True; continue
                        if start_table and line.strip() and not line.startswith('#'):
                            parts = line.split()
                            if len(parts) >= 5 and parts[4] in structures_list:
                                new_row[parts[4]] = float(parts[3])

                    df_final = pd.concat([df_final, pd.DataFrame([new_row])], ignore_index=True)
                    print(f"  SUCCESS: Added {sid} session with global measures.")
    except Exception as e:
        print(f"  Failed {sid}: {e}")

# 4. Final Sorting
df_final['PATNO'] = df_final['PATNO'].astype(str)
df_final['Study_Date_Dt'] = pd.to_datetime(df_final['Study_Date'], format='%m-%d-%y')
df_final = df_final.sort_values(by=['PATNO', 'Study_Date_Dt']).drop(columns=['Study_Date_Dt'])

# 5. Save
df_final.to_csv(master_path, index=False)
print(f"\\nFinal Row Count: {len(df_final)}")
display(df_final[df_final['PATNO'].isin(['3309', '3323'])][['PATNO', 'EVENT_ID', 'Study_Date', 'BrainSegVol']])

Extracting manual session for 3309 with corrected regex...
  SUCCESS: Added 3309 session with global measures.
Extracting manual session for 3323 with corrected regex...
  SUCCESS: Added 3323 session with global measures.
\nFinal Row Count: 1036


,PATNO,EVENT_ID,Study_Date,BrainSegVol
747,3309,BL,01-12-12,9.754473e+05
748,3309,V04,02-20-13,9.914564e+05
749,3309,V06,01-07-14,9.905753e+05
1034,3309,V08,12-15-15,9.937583e+05
755,3323,BL,07-19-12,1.354775e+06
756,3323,V04,08-15-13,1.329854e+06
1035,3323,V08,08-10-16,1.289793e+06
